# Seeing the Game: soccer player identification from still images (full pipeline)
**CSC 381/481 · Muzammil Ali Khan, Sidra Azeem**

This notebook runs the whole project top to bottom:

| Part | What it does | Original notebook |
|---|---|---|
| **I. Field, players and teams** | Grass mask, field region, player detection, team colours (+ Part 8 improvements) | `01_field_and_teams.ipynb` (Muzammil) |
| **II. Jersey numbers** | Torso → shirt colour → number mask → digits → classifier → "unreadable" (+ Part C improvements) | `02_jersey_numbers.ipynb` (Sidra) |
| **III. Both halves together** | Muzammil's detector boxes are read by Sidra's reader on the same 60 GSR frames | new |

**How to run (Colab):** Runtime → Run all. Part I streams 60 GSR frames from Hugging Face; Part II mounts Google
Drive and uses the jersey data and cache in `MyDrive/csc481_soccer` (first time ever: set `DOWNLOAD_DATA = True` in
Part II's settings cell). The two original notebooks are unchanged and still run on their own.

# Part I: Field, players and teams (Muzammil)

# Soccer Player Identification from Still Images
## Field and Team Side: separating grass from players and grouping players into teams

**What this notebook does.** Given a still image from a soccer match, it (1) finds the grass, (2) finds the people standing on it, (3) looks at the color of each person's shirt, and (4) groups the people into the two teams, the referees and the goalkeepers. It never uses a pretrained model. Every step is classical image processing from the course.

**How to read it.** Each part has a plain-language explanation first, then the code, and every code step has comments. The tags below tell you what each piece is for:

- **[PRESENTATION]** = this figure, table or idea is meant for the 4-5 minute video.
- **[ASSESSMENT]** = a question you could get on the paper assessment, with the answer in plain words.
- **[NEW CODE]** = something in the code that is new compared to your assignments. It is explained right where it appears.

**Style.** Like in the assignments, Part 3 works step by step on three example images (`image1`, `image2`, `image3`) with the code written out for each one. Parts 4 and 5 have to repeat the same steps on dozens of images and on many settings, so there I use a few **functions**. A function is just a block of code with a name, so we can reuse it instead of copy-pasting it 60 times (it is the same idea as writing the steps for `image1`, `image2` and `image3`, but for many more images).

**Where each part goes in the video (4-5 minutes)**

| Video section | Where to find it |
|---|---|
| Introduction, problem and why it matters | Part 0 text below |
| Example inputs and desired outputs | Part 2, ground-truth figure |
| Pipeline diagram | Part 0 |
| Intermediate outputs of each step | Part 3 |
| Libraries and tools used | Part 0, table below |
| Dataset description and statistics | Part 2, statistics cell |
| Preliminary results | Parts 4 and 5, results tables |
| Hyperparameters that affect the results | Part 4 (detection) and Part 5 (teams) |
| Success and failure examples | Part 6 |
| Sources of error, conclusion, future work | Part 7 |

**Libraries used (say this in the video and in the assessment)**

| Library | What we use it for | Own work or library? |
|---|---|---|
| OpenCV (`cv2`) | color conversion, morphology, connected components, convex hull, K-means | the algorithms are the library's; **choosing and combining the steps and all the settings is our own design** |
| NumPy | arrays and counting | library |
| Matplotlib | figures | library |
| `huggingface_hub`, `zipfile`, `json` | reading a few frames and labels out of the big dataset file | library |

In [ ]:
import os
import json
import glob
import zipfile
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2

# PART 0 - The Problem and the Pipeline

**The problem.** Photographers, media teams and analysts still mark players in match photos by hand, and cheap automatic tools are rare for amateur and youth teams. Our project identifies players in a still photo by team (shirt color) and by jersey number. *This notebook covers the first half: finding the people on the field and sorting them into teams, referees and goalkeepers.* My partner covers the jersey numbers.

**Why it is worth solving.** Doing this by hand for hundreds of photos is slow, and professional tracking systems are expensive.

**The idea in one sentence.** A soccer pitch is mostly green, so everything that is *not* green inside the pitch is a person (or a line). Then people wear shirts of different colors, so grouping the shirt colors gives us the teams.

## [PRESENTATION] Pipeline diagram

In [ ]:
#Pipeline diagram drawn with boxes and arrows
steps = [
    "1. Input\nstill image",
    "2. HSV grass\nmask",
    "3. Morphology:\nclean the grass",
    "4. Field region\n(convex hull)",
    "5. Non-grass pixels\ninside the field",
    "6. Morphology:\nremove pitch lines",
    "7. Blobs ->\nplayer boxes",
    "8. Torso crop ->\ndominant color",
    "9. K-means -> teams,\nreferees, goalkeepers",
]

#Position of each box (first row left to right, second row right to left)
xs = [1, 3, 5, 7, 9, 9, 7, 5, 3]
ys = [3, 3, 3, 3, 3, 1, 1, 1, 1]

#Box colors: grass steps, morphology steps, blob steps, color and team steps
colors = ["#d9d9d9", "#b7e1a1", "#ffd59a", "#b7e1a1", "#b7e1a1", "#ffd59a", "#a6cef2", "#d8b7ec", "#d8b7ec"]

plt.figure(figsize=(15, 5))
ax = plt.gca()
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.axis("off")

for i in range(len(steps)):
    ax.text(xs[i], ys[i], steps[i], ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.5", facecolor=colors[i], edgecolor="black"))

#Arrows between consecutive boxes
for i in range(len(steps) - 1):
    if ys[i] == ys[i + 1]:
        direction = 1 if xs[i + 1] > xs[i] else -1
        start = (xs[i] + direction * 0.95, ys[i])
        end = (xs[i + 1] - direction * 0.95, ys[i + 1])
    else:
        start = (xs[i], ys[i] - 0.4)
        end = (xs[i + 1], ys[i + 1] + 0.4)
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", lw=2))

plt.title("Field and team pipeline")
plt.tight_layout()
plt.show()

> **[ASSESSMENT] Pipeline in one breath (practice saying this out loud).**
> "I convert the image to HSV and threshold the green hue to get the grass. Morphological opening removes tiny grass specks, a closing reconnects the pieces that the pitch lines split, I keep the biggest grass piece and take its convex hull as the field region. Inside the field, anything that is not grass is a candidate: people, pitch lines, the ball. Opening removes the thin pitch lines, closing joins the pieces of one player. Connected components turn the remaining white regions into blobs and I keep the ones with a plausible size and shape as player boxes. For each box I crop the torso, keep the non-grass pixels, and take the most common quantized color as the shirt color. Finally K-means groups those colors into teams, referees and goalkeepers."

# PART 1 - Getting the Data

**Dataset.** SoccerNet Game State Reconstruction (SoccerNet-GSR). It has 30-second broadcast clips and, for every frame, a box around each person with a role (player, goalkeeper, referee, other), a team (left or right) and sometimes a jersey number.

**Why this dataset.** It is real broadcast footage with labels for exactly what we need, so we can measure our results. It comes from the benchmark used in the papers of my literature review.

**The problem with the data.** The whole validation set is one file of about 11 GB. We do not need all of it for a classical pipeline with no training. So instead of downloading it, the code reads the table of contents of the file over the internet and takes only a few clips: their label files and a handful of frames.

**How the sample is made (the sub-sampling procedure).**
1. List all the clips in the file, shuffle them with a fixed random seed (so anyone gets the same clips) and take the first few. This avoids picking clips by hand, so we do not choose only the easy ones.
2. In each clip take frames evenly spaced in time, so the frames are far apart and are not nearly the same picture. Each frame is then treated as an **independent still image**, as promised in the proposal.
3. Save the frames and a small label file to disk, so the next runs do not use the internet.

> **[ASSESSMENT] Data collection and fitness.**
> *How did you collect the data?* I sub-sampled SoccerNet-GSR: random clips (fixed seed) and evenly spaced frames from each, read directly from the dataset file.
> *Why is it appropriate?* It is real broadcast footage with player boxes, roles and teams, which is exactly what I need for evaluation.
> *Is the sample representative?* It covers several different matches (different stadiums, kits and lighting), but it is small and only from the validation split, so results are indicative and not a full benchmark.

In [ ]:
#Settings for sampling the data (change these to get more or fewer images)
data_dir = "sampled_data"          #folder where the sampled frames and labels are saved
num_sequences = 6                  #how many 30-second clips are used
frames_per_sequence = 10           #how many frames are taken from each clip

os.makedirs(data_dir, exist_ok=True)

#Clips that were already saved in a previous run
saved_labels = glob.glob(os.path.join(data_dir, "*", "labels.json"))
print("Clips already saved:", len(saved_labels), "of", num_sequences)

In [ ]:
#This cell only uses the internet if some clips are still missing
if len(saved_labels) >= num_sequences:
    print("Nothing to download, using the saved data")
else:
    #Longer waiting time for slow connections (must be set before importing huggingface_hub)
    os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
    from huggingface_hub import HfFileSystem

    #Opening the remote zip file. Nothing big is downloaded here: Python only asks for the parts it needs
    fs = HfFileSystem()
    remote_file = fs.open("datasets/SoccerNet/SN-GSR-2024/valid.zip", "rb", block_size=2 * 1024 * 1024)
    z = zipfile.ZipFile(remote_file)
    names = z.namelist()             #list of all the files inside the zip
    name_set = set(names)            #same list as a set, which is much faster to search

    #There is one label file per clip, so the label files give us the list of clips
    label_names = []
    for n in names:
        if n.endswith("Labels-GameState.json"):
            label_names.append(n)
    label_names = sorted(label_names)
    print("Clips available in the zip:", len(label_names))

    #Shuffling with a fixed seed and taking the first ones (same clips every time)
    random.seed(481)
    random.shuffle(label_names)
    chosen = label_names[:num_sequences]

    for label_name in chosen:
        seq_folder = label_name.rsplit("/", 1)[0]       #folder of the clip inside the zip
        seq_name = os.path.basename(seq_folder)         #name of the clip, for example SNGS-021
        seq_dir = os.path.join(data_dir, seq_name)      #where we save it on disk

        if os.path.exists(os.path.join(seq_dir, "labels.json")):
            print("Already saved:", seq_name)
            continue

        print("Downloading:", seq_name)
        try:
            os.makedirs(os.path.join(seq_dir, "frames"), exist_ok=True)

            #Only the bytes of this one label file are downloaded
            labels = json.loads(z.read(label_name))

            #Frames in time order, then evenly spaced frames
            images = sorted(labels["images"], key=lambda im: im["file_name"])
            step = max(1, len(images) // frames_per_sequence)

            saved = []
            for img in images[::step][:frames_per_sequence]:
                #Boxes of the people in this frame (the other annotations are the pitch lines and the camera)
                boxes = []
                for a in labels["annotations"]:
                    if a["supercategory"] == "object" and a["image_id"] == img["image_id"]:
                        box = {
                            "role": a["attributes"].get("role"),
                            "team": a["attributes"].get("team"),
                            "jersey": a["attributes"].get("jersey"),
                            "x": a["bbox_image"]["x"], "y": a["bbox_image"]["y"],
                            "w": a["bbox_image"]["w"], "h": a["bbox_image"]["h"],
                        }
                        boxes.append(box)
                if len(boxes) == 0:
                    continue

                #Frames are inside the img1 folder of the clip
                frame_path = seq_folder + "/img1/" + img["file_name"]
                if frame_path not in name_set:
                    frame_path = seq_folder + "/" + img["file_name"]

                file_name = os.path.basename(img["file_name"])
                with open(os.path.join(seq_dir, "frames", file_name), "wb") as f:
                    f.write(z.read(frame_path))         #only the bytes of this one frame are downloaded
                saved.append({"file_name": file_name, "boxes": boxes})

            #labels.json is written last, so a clip only counts as saved when all its frames are saved
            with open(os.path.join(seq_dir, "labels.json"), "w") as f:
                json.dump(saved, f)
            print("   saved", len(saved), "frames")

        except Exception as e:
            print("Failed:", seq_name, "-", e, "(run this cell again to retry)")

# PART 2 - Looking at the Data

Loading the saved frames and labels into one list called `frames`. Each item of the list is a **dictionary**, which stores values under names, so `f["image"]` is the image of that frame and `f["boxes"]` are its ground-truth boxes.

For every person we also store a **label**, the group the dataset says the person belongs to (for example `player_left` or `referee`). We use it later only to measure our results.

In [ ]:
#Loading all the saved frames
frames = []
seq_names = []          #names of the clips, in order

for labels_path in sorted(glob.glob(os.path.join(data_dir, "*", "labels.json"))):
    seq_dir = os.path.dirname(labels_path)
    seq_name = os.path.basename(seq_dir)
    seq_names.append(seq_name)

    with open(labels_path) as f:
        saved = json.load(f)

    for item in saved:
        image = cv2.imread(os.path.join(seq_dir, "frames", item["file_name"]))
        if image is None:
            continue

        #Adding the group (label) to every ground-truth box
        people = []              #players, goalkeepers and referees (the "other" people are not part of our task)
        for b in item["boxes"]:
            if b["role"] == "player":
                b["label"] = "player_" + str(b["team"])
            elif b["role"] == "goalkeeper":
                b["label"] = "goalkeeper_" + str(b["team"])
            else:
                b["label"] = b["role"]          #referee or other

            if b["role"] in ["player", "goalkeeper", "referee"]:
                people.append(b)

        frames.append({"name": seq_name + "_" + item["file_name"], "seq": seq_name, "clip_id": len(seq_names) - 1,
                       "image": image, "boxes": item["boxes"], "people": people})

print("Frames loaded:", len(frames))

## [PRESENTATION] Dataset statistics

Ballpark numbers are enough for the assessment, but know the pattern: *a few clips, tens of frames, 20-30 people per frame, and many small players.*

In [ ]:
#Dataset statistics
print("Clips (different matches):", len(seq_names))
print("Frames:", len(frames))
print("Image size (height, width, channels):", frames[0]["image"].shape)

#Number of boxes for each role, and players for each team
count_player = 0
count_goalkeeper = 0
count_referee = 0
count_other = 0
left_players = 0
right_players = 0
people_per_frame = []
player_heights = []

for f in frames:
    people_per_frame.append(len(f["boxes"]))
    for b in f["boxes"]:
        if b["role"] == "player":
            count_player += 1
            player_heights.append(b["h"])
            if b["team"] == "left":
                left_players += 1
            else:
                right_players += 1
        elif b["role"] == "goalkeeper":
            count_goalkeeper += 1
        elif b["role"] == "referee":
            count_referee += 1
        else:
            count_other += 1

print("Players:", count_player, ", goalkeepers:", count_goalkeeper, ", referees:", count_referee, ", others:", count_other)
print("Players - left team:", left_players, ", right team:", right_players)
print("People per frame: min", min(people_per_frame), ", median", int(np.median(people_per_frame)), ", max", max(people_per_frame))
print("Player box height in pixels: min", int(min(player_heights)), ", median", int(np.median(player_heights)), ", max", int(max(player_heights)))

#How tall are the players? Small players are hard to work with
plt.figure(figsize=(8, 4))
plt.hist(player_heights, bins=30)
plt.title("Height of the player boxes (pixels)")
plt.xlabel("Box height")
plt.ylabel("Number of players")
plt.tight_layout()
plt.show()

## [PRESENTATION] Example inputs and desired outputs

Three input frames from different matches (top row) and what we want to get (bottom row): a box around every person, colored by group. The bottom row is the **ground truth** from the dataset: red and blue = the two teams (left and right), yellow = goalkeepers, magenta = referees, white = other people.

From here on, `image1`, `image2` and `image3` are these three example frames (the middle frame of the first three clips).

In [ ]:
#Choosing the middle frame of the first three clips as the example images
demo_ids = []
for c in range(3):
    ids = []
    for i in range(len(frames)):
        if frames[i]["clip_id"] == c:
            ids.append(i)
    demo_ids.append(ids[len(ids) // 2])

image1 = frames[demo_ids[0]]["image"]
image2 = frames[demo_ids[1]]["image"]
image3 = frames[demo_ids[2]]["image"]

#Converting BGR to RGB for display
image1_rgb = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
image2_rgb = cv2.cvtColor(image2, cv2.COLOR_BGR2RGB)
image3_rgb = cv2.cvtColor(image3, cv2.COLOR_BGR2RGB)
demo_rgb = [image1_rgb, image2_rgb, image3_rgb]

#Colors (RGB) for each group of people in the ground truth
gt_colours = {
    "player_left": (255, 0, 0),
    "player_right": (0, 0, 255),
    "goalkeeper_left": (255, 255, 0),
    "goalkeeper_right": (255, 255, 0),
    "referee": (255, 0, 255),
    "other": (255, 255, 255),
}

#Drawing the ground-truth boxes on a copy of each example image
gt_drawn = []
for k in range(3):
    drawn = demo_rgb[k].copy()
    for b in frames[demo_ids[k]]["boxes"]:
        x = int(b["x"])
        y = int(b["y"])
        w = int(b["w"])
        h = int(b["h"])
        cv2.rectangle(drawn, (x, y), (x + w, y + h), gt_colours.get(b["label"], (255, 255, 255)), 3)
    gt_drawn.append(drawn)

plt.figure(figsize=(15, 8))

for k in range(3):
    plt.subplot(2, 3, k + 1)
    plt.imshow(demo_rgb[k])
    plt.title("Input " + str(k + 1))
    plt.axis("off")

    plt.subplot(2, 3, k + 4)
    plt.imshow(gt_drawn[k])
    plt.title("Desired output " + str(k + 1) + " (ground truth)")
    plt.axis("off")

plt.tight_layout()
plt.show()

> **[ASSESSMENT] Dataset questions.**
> *Ballpark size?* Around 6 clips and 60 frames, with about 20-30 people per frame, so well over a thousand labeled people.
> *Class distribution?* Most boxes are players, split roughly evenly between the two teams; goalkeepers and referees are a small minority.
> *Noise in the images?* Broadcast footage: motion blur when the camera pans, shadows and uneven light on the grass, players overlapping each other, many players that are tiny because they are far from the camera, and advertising boards and crowds around the pitch.
> *Why is there no training data?* This half of the project uses no learning: color thresholds, morphology and K-means need no labeled training set. The labels are only used to **measure** how well it works.

# PART 3 - The Pipeline Step by Step (on the 3 example images)

Every step is shown on the same three images (`image1`, `image2`, `image3`), so you can see what each step does. In Part 4 the same steps are packaged into a function to run them on all the frames.

# (a) Step 1 - Finding the grass with HSV

**Why not RGB?** In RGB, the color of the grass and how bright it is are mixed together. The grass on a pitch has bright parts, shadows and mowing stripes, so its RGB values change a lot. In **HSV** the color is separated from the brightness:

- **H (hue)** = which color it is. In OpenCV it goes from 0 to 179. Green is around 60.
- **S (saturation)** = how pure the color is. Low saturation = grey or white.
- **V (value)** = how bright it is.

The grass keeps almost the same hue even in shadow. So we select pixels whose hue is in the green range, and we also require some minimum saturation and brightness, so that white lines, grey crowds and very dark areas are not selected.

In [ ]:
#Looking at the H, S and V channels of image 1
hsv1 = cv2.cvtColor(image1, cv2.COLOR_BGR2HSV)

plt.figure(figsize=(15, 8))

plt.subplot(2, 2, 1)
plt.imshow(image1_rgb)
plt.title("Image 1 - Input")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(hsv1[:, :, 0], cmap="gray")
plt.title("H (hue): the grass is a uniform grey area")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(hsv1[:, :, 1], cmap="gray")
plt.title("S (saturation)")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(hsv1[:, :, 2], cmap="gray")
plt.title("V (value, brightness)")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#Grass thresholds (OpenCV: H in 0-179, S and V in 0-255). These are the settings we will test later
hue_low = 35
hue_high = 85
sat_min = 40
val_min = 40

#Grass mask for each image: white (255) = grass, black (0) = not grass
hsv1 = cv2.cvtColor(image1, cv2.COLOR_BGR2HSV)
hsv2 = cv2.cvtColor(image2, cv2.COLOR_BGR2HSV)
hsv3 = cv2.cvtColor(image3, cv2.COLOR_BGR2HSV)

grass_raw1 = ((hsv1[:, :, 0] >= hue_low) & (hsv1[:, :, 0] <= hue_high) & (hsv1[:, :, 1] >= sat_min) & (hsv1[:, :, 2] >= val_min)).astype(np.uint8) * 255
grass_raw2 = ((hsv2[:, :, 0] >= hue_low) & (hsv2[:, :, 0] <= hue_high) & (hsv2[:, :, 1] >= sat_min) & (hsv2[:, :, 2] >= val_min)).astype(np.uint8) * 255
grass_raw3 = ((hsv3[:, :, 0] >= hue_low) & (hsv3[:, :, 0] <= hue_high) & (hsv3[:, :, 1] >= sat_min) & (hsv3[:, :, 2] >= val_min)).astype(np.uint8) * 255

plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
plt.imshow(image1_rgb)
plt.title("Image 1 - Input")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(image2_rgb)
plt.title("Image 2 - Input")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(image3_rgb)
plt.title("Image 3 - Input")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(grass_raw1, cmap="gray")
plt.title("Image 1 - Grass mask (raw)")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(grass_raw2, cmap="gray")
plt.title("Image 2 - Grass mask (raw)")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(grass_raw3, cmap="gray")
plt.title("Image 3 - Grass mask (raw)")
plt.axis("off")

plt.tight_layout()
plt.show()

**What to notice.** The players show up as black "holes" inside the white grass area. The pitch lines are also black thin lines. There may be small white specks outside the pitch (green things in the crowd or advertising). Next we clean those.

### [PRESENTATION] How the grass settings change the mask
The hue range and the minimum saturation are the first settings that affect the result. A range that is too narrow loses parts of the grass (sunny or shaded areas); a range that is too wide starts to include other colors.

In [ ]:
#Trying different grass settings on image 1: (hue low, hue high, minimum saturation)
settings = [(35, 85, 40), (30, 90, 40), (45, 75, 40), (35, 85, 20), (35, 85, 80), (50, 70, 40)]

plt.figure(figsize=(15, 8))

for k in range(len(settings)):
    h_low = settings[k][0]
    h_high = settings[k][1]
    s_min = settings[k][2]
    mask = ((hsv1[:, :, 0] >= h_low) & (hsv1[:, :, 0] <= h_high) & (hsv1[:, :, 1] >= s_min) & (hsv1[:, :, 2] >= val_min)).astype(np.uint8) * 255

    plt.subplot(2, 3, k + 1)
    plt.imshow(mask, cmap="gray")
    plt.title("Hue " + str(h_low) + "-" + str(h_high) + ", min saturation " + str(s_min))
    plt.axis("off")

plt.tight_layout()
plt.show()

> **[ASSESSMENT] Grass mask.**
> *Why HSV?* It separates color (hue) from brightness, so grass in sun and in shadow still has about the same hue. In RGB all three channels change with brightness.
> *What do the thresholds do?* Hue 35-85 keeps green. The minimum saturation removes whites and greys (lines, crowd, overcast areas). The minimum value removes very dark pixels.
> *How did you choose them?* By looking at the masks on several frames (the figure above) and then measuring detection quality for a few settings (Part 4).
> *What goes wrong?* Pitches with yellowish or very dark grass, strong shadows, and players wearing green (their shirt is classified as grass).

# (b) Step 2 - Cleaning the grass mask and finding the field region

Four operations, each from the morphology lecture:

1. **Opening** (erosion followed by dilation) with a small elliptical structuring element removes the tiny grass specks outside the pitch. Opening removes small objects and keeps the big ones.
2. **Closing** (dilation followed by erosion) with a bigger structuring element **only to find the pitch**. The white pitch lines (for example the halfway line) cut across the whole pitch and split the grass into several pieces. Closing is wider than a line, so it reconnects those pieces.
3. **Largest connected component.** After the closing, the pitch is by far the biggest piece, so we keep only that one.
4. **Convex hull.** The players are holes inside the grass. The convex hull is the smallest convex shape that contains the whole grass region, so it fills those holes (and the lines). The result is the **field region**: everything inside the pitch, grass or not.

Important: the closing is used only to locate the pitch. The grass mask we keep for the next steps is the one after the opening, so the players are still holes in it.

In [ ]:
#Elliptical structuring elements: a small one for the opening, a big one (wider than a pitch line) for the closing
kernel_grass = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
kernel_pitch = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))

#1. Opening removes the tiny specks
grass_open1 = cv2.morphologyEx(grass_raw1, cv2.MORPH_OPEN, kernel_grass)
grass_open2 = cv2.morphologyEx(grass_raw2, cv2.MORPH_OPEN, kernel_grass)
grass_open3 = cv2.morphologyEx(grass_raw3, cv2.MORPH_OPEN, kernel_grass)

#2. Closing reconnects the grass pieces that the pitch lines split apart
grass_joined1 = cv2.morphologyEx(grass_open1, cv2.MORPH_CLOSE, kernel_pitch)
grass_joined2 = cv2.morphologyEx(grass_open2, cv2.MORPH_CLOSE, kernel_pitch)
grass_joined3 = cv2.morphologyEx(grass_open3, cv2.MORPH_CLOSE, kernel_pitch)

#3. Connected components: label 0 is the background, the others are the grass pieces.
#stats[i, cv2.CC_STAT_AREA] is the number of pixels of piece i. We keep the piece with the biggest area
num1, labels1, stats1, centroids1 = cv2.connectedComponentsWithStats(grass_joined1, connectivity=8)
num2, labels2, stats2, centroids2 = cv2.connectedComponentsWithStats(grass_joined2, connectivity=8)
num3, labels3, stats3, centroids3 = cv2.connectedComponentsWithStats(grass_joined3, connectivity=8)

biggest1 = 1 + np.argmax(stats1[1:, cv2.CC_STAT_AREA])      #[1:] skips the background
biggest2 = 1 + np.argmax(stats2[1:, cv2.CC_STAT_AREA])
biggest3 = 1 + np.argmax(stats3[1:, cv2.CC_STAT_AREA])

pitch1 = (labels1 == biggest1).astype(np.uint8) * 255
pitch2 = (labels2 == biggest2).astype(np.uint8) * 255
pitch3 = (labels3 == biggest3).astype(np.uint8) * 255

#4. Convex hull: first the outline (contour) of the pitch, then the hull of that outline.
#[NEW CODE] max(contours, key=cv2.contourArea) picks the contour with the biggest area
contours1, _ = cv2.findContours(pitch1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours2, _ = cv2.findContours(pitch2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours3, _ = cv2.findContours(pitch3, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

hull1 = cv2.convexHull(max(contours1, key=cv2.contourArea))
hull2 = cv2.convexHull(max(contours2, key=cv2.contourArea))
hull3 = cv2.convexHull(max(contours3, key=cv2.contourArea))

#Filling the hull with white to get the field region mask
field1 = np.zeros_like(grass_open1)
field2 = np.zeros_like(grass_open2)
field3 = np.zeros_like(grass_open3)
cv2.fillConvexPoly(field1, hull1, 255)
cv2.fillConvexPoly(field2, hull2, 255)
cv2.fillConvexPoly(field3, hull3, 255)

#Displaying the steps for the three images
plt.figure(figsize=(15, 12))

plt.subplot(3, 4, 1)
plt.imshow(grass_raw1, cmap="gray")
plt.title("Image 1 - Raw grass mask")
plt.axis("off")

plt.subplot(3, 4, 2)
plt.imshow(grass_open1, cmap="gray")
plt.title("Image 1 - After opening")
plt.axis("off")

plt.subplot(3, 4, 3)
plt.imshow(grass_joined1, cmap="gray")
plt.title("Image 1 - After closing (pieces rejoined)")
plt.axis("off")

plt.subplot(3, 4, 4)
plt.imshow(field1, cmap="gray")
plt.title("Image 1 - Field region (convex hull)")
plt.axis("off")

plt.subplot(3, 4, 5)
plt.imshow(grass_raw2, cmap="gray")
plt.title("Image 2 - Raw grass mask")
plt.axis("off")

plt.subplot(3, 4, 6)
plt.imshow(grass_open2, cmap="gray")
plt.title("Image 2 - After opening")
plt.axis("off")

plt.subplot(3, 4, 7)
plt.imshow(grass_joined2, cmap="gray")
plt.title("Image 2 - After closing")
plt.axis("off")

plt.subplot(3, 4, 8)
plt.imshow(field2, cmap="gray")
plt.title("Image 2 - Field region")
plt.axis("off")

plt.subplot(3, 4, 9)
plt.imshow(grass_raw3, cmap="gray")
plt.title("Image 3 - Raw grass mask")
plt.axis("off")

plt.subplot(3, 4, 10)
plt.imshow(grass_open3, cmap="gray")
plt.title("Image 3 - After opening")
plt.axis("off")

plt.subplot(3, 4, 11)
plt.imshow(grass_joined3, cmap="gray")
plt.title("Image 3 - After closing")
plt.axis("off")

plt.subplot(3, 4, 12)
plt.imshow(field3, cmap="gray")
plt.title("Image 3 - Field region")
plt.axis("off")

plt.tight_layout()
plt.show()

**What to notice.** The field region should cover the pitch and not the stands or the sky. If the camera shows a corner of the pitch that is not convex, or a big grass-colored object appears elsewhere, the hull can be wrong. That is a failure case worth looking for.

> **[ASSESSMENT] Morphology and hull.**
> *What is opening and why here?* Erosion then dilation. Erosion shrinks white regions and removes anything smaller than the structuring element; dilation then grows the survivors back. So small specks disappear and the big pitch stays almost unchanged.
> *Why a closing before finding the biggest piece?* The white lines (the halfway line, for example) cross the whole pitch and cut the grass mask into several pieces. A closing wider than a line reconnects them, so the biggest piece is the whole pitch and not just one part of it. It is used only to find the pitch: the grass mask used afterwards is the one without the closing.
> *Why the convex hull?* The people are holes in the grass mask. The hull fills those holes in one step and gives us the area where people can be.
> *Why not use a closing to fill the players?* A closing big enough to fill a player would also change the shape of the pitch, and players have different sizes. The hull does not depend on player size.

# (c) Step 3 - People = non-grass pixels inside the field

Inside the field region, every pixel that is not grass belongs to a player, a referee, the ball or a pitch line. So:

**non-grass inside the field = field region AND NOT grass**

Then two more morphology steps:

- **Opening with a bigger structuring element** removes the thin pitch lines (they are only a few pixels wide) and tiny noise, but players are thick enough to survive.
- **Closing with a small structuring element** joins the pieces of the same player that came apart (for example, a gap between the shirt and the shorts).

In [ ]:
kernel_people_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
kernel_people_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

#Inside the field AND NOT grass (bitwise_not turns white into black and black into white)
non_grass_raw1 = cv2.bitwise_and(field1, cv2.bitwise_not(grass_open1))
non_grass_raw2 = cv2.bitwise_and(field2, cv2.bitwise_not(grass_open2))
non_grass_raw3 = cv2.bitwise_and(field3, cv2.bitwise_not(grass_open3))

#Opening removes the thin pitch lines and noise
non_grass_open1 = cv2.morphologyEx(non_grass_raw1, cv2.MORPH_OPEN, kernel_people_open)
non_grass_open2 = cv2.morphologyEx(non_grass_raw2, cv2.MORPH_OPEN, kernel_people_open)
non_grass_open3 = cv2.morphologyEx(non_grass_raw3, cv2.MORPH_OPEN, kernel_people_open)

#Closing joins the pieces of the same player
non_grass_clean1 = cv2.morphologyEx(non_grass_open1, cv2.MORPH_CLOSE, kernel_people_close)
non_grass_clean2 = cv2.morphologyEx(non_grass_open2, cv2.MORPH_CLOSE, kernel_people_close)
non_grass_clean3 = cv2.morphologyEx(non_grass_open3, cv2.MORPH_CLOSE, kernel_people_close)

plt.figure(figsize=(15, 12))

plt.subplot(3, 3, 1)
plt.imshow(non_grass_raw1, cmap="gray")
plt.title("Image 1 - Non-grass inside the field")
plt.axis("off")

plt.subplot(3, 3, 2)
plt.imshow(non_grass_open1, cmap="gray")
plt.title("Image 1 - After opening (lines removed)")
plt.axis("off")

plt.subplot(3, 3, 3)
plt.imshow(non_grass_clean1, cmap="gray")
plt.title("Image 1 - After closing")
plt.axis("off")

plt.subplot(3, 3, 4)
plt.imshow(non_grass_raw2, cmap="gray")
plt.title("Image 2 - Non-grass inside the field")
plt.axis("off")

plt.subplot(3, 3, 5)
plt.imshow(non_grass_open2, cmap="gray")
plt.title("Image 2 - After opening")
plt.axis("off")

plt.subplot(3, 3, 6)
plt.imshow(non_grass_clean2, cmap="gray")
plt.title("Image 2 - After closing")
plt.axis("off")

plt.subplot(3, 3, 7)
plt.imshow(non_grass_raw3, cmap="gray")
plt.title("Image 3 - Non-grass inside the field")
plt.axis("off")

plt.subplot(3, 3, 8)
plt.imshow(non_grass_open3, cmap="gray")
plt.title("Image 3 - After opening")
plt.axis("off")

plt.subplot(3, 3, 9)
plt.imshow(non_grass_clean3, cmap="gray")
plt.title("Image 3 - After closing")
plt.axis("off")

plt.tight_layout()
plt.show()

# (d) Step 4 - From white regions to player boxes

**Connected components** again: each white region becomes one blob and OpenCV gives us its bounding box and its area. Then we keep only blobs that look like a person:

- not too small (area and height), to drop noise and far-away specks,
- not too big (a blob bigger than 5% of the image is not a player),
- taller than wide (height/width between 0.8 and 5), since a standing person is a tall shape.

Players who touch or overlap end up in **one** blob. That is a known weakness we will see in the failure cases.

The boxes we draw here are in **red**; the ground truth is in thin **white**.

In [ ]:
#Settings for choosing the blobs that look like people
min_area = 300
min_height = 20
min_aspect = 0.8       #height / width
max_aspect = 5.0
max_area_fraction = 0.05

#Connected components of the cleaned masks
num1, labels1, stats1, centroids1 = cv2.connectedComponentsWithStats(non_grass_clean1, connectivity=8)
num2, labels2, stats2, centroids2 = cv2.connectedComponentsWithStats(non_grass_clean2, connectivity=8)
num3, labels3, stats3, centroids3 = cv2.connectedComponentsWithStats(non_grass_clean3, connectivity=8)

#Image 1: keeping only the blobs that look like a person
boxes1 = []
img_h1, img_w1 = non_grass_clean1.shape
for i in range(1, num1):                         #label 0 is the background
    x = int(stats1[i, cv2.CC_STAT_LEFT])
    y = int(stats1[i, cv2.CC_STAT_TOP])
    w = int(stats1[i, cv2.CC_STAT_WIDTH])
    h = int(stats1[i, cv2.CC_STAT_HEIGHT])
    area = int(stats1[i, cv2.CC_STAT_AREA])

    if area < min_area or h < min_height:
        continue
    if area > max_area_fraction * img_h1 * img_w1:
        continue
    if h / w < min_aspect or h / w > max_aspect:
        continue
    boxes1.append((x, y, w, h))

#Image 2
boxes2 = []
img_h2, img_w2 = non_grass_clean2.shape
for i in range(1, num2):
    x = int(stats2[i, cv2.CC_STAT_LEFT])
    y = int(stats2[i, cv2.CC_STAT_TOP])
    w = int(stats2[i, cv2.CC_STAT_WIDTH])
    h = int(stats2[i, cv2.CC_STAT_HEIGHT])
    area = int(stats2[i, cv2.CC_STAT_AREA])

    if area < min_area or h < min_height:
        continue
    if area > max_area_fraction * img_h2 * img_w2:
        continue
    if h / w < min_aspect or h / w > max_aspect:
        continue
    boxes2.append((x, y, w, h))

#Image 3
boxes3 = []
img_h3, img_w3 = non_grass_clean3.shape
for i in range(1, num3):
    x = int(stats3[i, cv2.CC_STAT_LEFT])
    y = int(stats3[i, cv2.CC_STAT_TOP])
    w = int(stats3[i, cv2.CC_STAT_WIDTH])
    h = int(stats3[i, cv2.CC_STAT_HEIGHT])
    area = int(stats3[i, cv2.CC_STAT_AREA])

    if area < min_area or h < min_height:
        continue
    if area > max_area_fraction * img_h3 * img_w3:
        continue
    if h / w < min_aspect or h / w > max_aspect:
        continue
    boxes3.append((x, y, w, h))

demo_boxes = [boxes1, boxes2, boxes3]

#Drawing our boxes (red) and the ground truth (thin white)
plt.figure(figsize=(15, 8))

for k in range(3):
    drawn = demo_rgb[k].copy()
    for b in frames[demo_ids[k]]["boxes"]:
        cv2.rectangle(drawn, (int(b["x"]), int(b["y"])), (int(b["x"] + b["w"]), int(b["y"] + b["h"])), (255, 255, 255), 1)
    for (x, y, w, h) in demo_boxes[k]:
        cv2.rectangle(drawn, (x, y), (x + w, y + h), (255, 0, 0), 3)

    plt.subplot(1, 3, k + 1)
    plt.imshow(drawn)
    plt.title("Image " + str(k + 1) + ": " + str(len(demo_boxes[k])) + " boxes found, " + str(len(frames[demo_ids[k]]["boxes"])) + " people labeled")
    plt.axis("off")

plt.tight_layout()
plt.show()

> **[ASSESSMENT] Detection steps.**
> *What are connected components?* Groups of touching foreground pixels; each gets its own label, and OpenCV reports each group's bounding box and area.
> *Why do you filter blobs?* After the morphology there can still be non-person blobs (the ball, a piece of a line, a large shadow). Size and shape rules remove most of them.
> *What is the main weakness?* Players who touch become one blob, and tiny far-away players are removed with the noise.
> *Is this a learned detector?* No. It uses only color, morphology and simple geometry rules.

# (e) Step 5 - The shirt color of each person

For each box we need one color that represents the shirt.

1. **Torso crop.** In the box, the head is at the top, then the torso, then shorts and legs. We take the part from 15% to 50% of the box height, and the middle 60% of the width (to avoid arms and background at the sides). This is a rule of thumb for how a person's body is proportioned.
2. **Remove grass pixels.** A crop can include a bit of grass behind the player; those pixels are ignored using the grass mask.
3. **Dominant color.** The shirt may have several colors (stripes, sponsor logos, numbers), so the average would mix them into a color that is not on the shirt. Instead, each color channel is quantized into 8 levels (so there are only 8 x 8 x 8 = 512 possible colors) and we take the **most common** one. This is the "dominant color by quantized mode" idea from the color analysis lectures.

Here it is done step by step for the people found in image 1 (up to 8 people).

In [ ]:
#Shirt colors of the people found in image 1
shown = boxes1[:8]
drawn = image1_rgb.copy()
crops = []
swatches = []

for (x, y, w, h) in shown:
    #1. Torso crop: 15%-50% of the height and the middle 60% of the width
    tx1 = int(x + 0.2 * w)
    tx2 = int(x + 0.8 * w)
    ty1 = int(y + 0.15 * h)
    ty2 = int(y + 0.5 * h)
    cv2.rectangle(drawn, (tx1, ty1), (tx2, ty2), (0, 255, 255), 3)         #torso area in cyan

    crop = image1[ty1:ty2, tx1:tx2]
    crop_grass = grass_open1[ty1:ty2, tx1:tx2]
    crops.append(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

    #2. Keeping only the pixels that are not grass (a list of pixels, each one with 3 values B, G, R)
    pixels = crop[crop_grass == 0]

    #3. Dominant color: each channel goes from 0-255 to 0-7 (levels of 32), and the three levels
    #are combined into one number from 0 to 511 (astype(int) is needed because the numbers get bigger than 255)
    q = pixels.astype(int) // 32
    ids = q[:, 0] * 64 + q[:, 1] * 8 + q[:, 2]

    #[NEW CODE] np.bincount counts how many pixels got each number. np.argmax gives the most common one
    counts = np.bincount(ids, minlength=512)
    best = int(np.argmax(counts))

    #Center of the winning color (back to a normal 0-255 color: B, G, R)
    b_val = (best // 64) * 32 + 16
    g_val = ((best // 8) % 8) * 32 + 16
    r_val = (best % 8) * 32 + 16

    #Small square of that color (as RGB) to display it
    swatch = np.zeros((20, 20, 3), dtype=np.uint8)
    swatch[:, :] = [r_val, g_val, b_val]
    swatches.append(swatch)

plt.figure(figsize=(15, 6))
plt.imshow(drawn)
plt.title("Image 1 - Torso areas (cyan)")
plt.axis("off")
plt.show()

plt.figure(figsize=(15, 4))
for k in range(len(shown)):
    plt.subplot(2, 8, k + 1)
    plt.imshow(crops[k])
    plt.title("Torso " + str(k + 1), fontsize=9)
    plt.axis("off")

    plt.subplot(2, 8, 8 + k + 1)
    plt.imshow(swatches[k])
    plt.title("Dominant color", fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

> **[ASSESSMENT] Shirt color.**
> *Why the torso?* The torso is where the shirt color is; the head is skin and hair, the legs are shorts and socks (a different color).
> *Why remove grass pixels?* Part of the crop can be the grass behind the player; if we kept it, every shirt would look a bit green.
> *Why dominant color and not the average?* Averaging a red shirt with white stripes gives pink, a color that is not on the shirt. The most common quantized color is a real color on the shirt.
> *What are the weak points?* Small players have a very small torso with few pixels, motion blur mixes colors, and two overlapping players share a blob.

# PART 4 - Running the Pipeline on All Frames and Measuring Detection

Steps 1-4 of Part 3 are now packaged into one **function**, `find_people`. It is **the same code** as in Part 3, just wrapped with a name so we can run it on every frame and test different settings without copy-pasting it dozens of times. Then we measure how good the detection is using the ground-truth boxes.

**How we measure (explained from zero).**

- **IoU (intersection over union)** compares two boxes: the area where they overlap divided by the area they cover together. 1 = identical, 0 = no overlap. We say our box *matches* a ground-truth person if IoU is at least 0.3 (a fairly generous value, because our blobs are tight around the body while labeled boxes can be looser).
- **Recall** = of all the people in the ground truth, what fraction did we find? (Misses lower it.)
- **Precision** = of all the boxes we drew, what fraction are really people? (False alarms lower it.)
- Both matter: drawing a box on everything gives high recall and bad precision; drawing very few boxes gives the opposite.

The ground truth is used **only to measure**. The pipeline itself never sees the labels.

In [ ]:
#Settings of the pipeline (the same values used in Part 3)
hue_low = 35
hue_high = 85
sat_min = 40
val_min = 40
open_grass = 5          #opening size for the grass mask
close_pitch = 25        #closing size to rejoin the grass pieces split by the pitch lines
open_people = 7         #opening size to remove the pitch lines from the people mask
close_people = 5        #closing size to join the pieces of a person
min_area = 300
min_height = 20
min_aspect = 0.8
max_aspect = 5.0
max_area_fraction = 0.05

#[NEW CODE] A function is a block of code with a name. The values in the parentheses (image_bgr, h_low, ...) are
#what we give to it, and "return" gives the results back. This one repeats Steps 1-4 of Part 3 and returns the grass mask,
#the field region, the cleaned non-grass mask and the list of boxes (x, y, w, h).
#The four values we will want to change in the tests (h_low, h_high, s_min, k_open) are inputs; the rest are the settings above.
def find_people(image_bgr, h_low, h_high, s_min, k_open):
    #Step 1: grass mask
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    grass = ((hsv[:, :, 0] >= h_low) & (hsv[:, :, 0] <= h_high) & (hsv[:, :, 1] >= s_min) & (hsv[:, :, 2] >= val_min)).astype(np.uint8) * 255

    #Step 2: opening, closing to rejoin the pieces, biggest grass piece, convex hull
    k_grass = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_grass, open_grass))
    grass = cv2.morphologyEx(grass, cv2.MORPH_OPEN, k_grass)

    k_pitch = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_pitch, close_pitch))
    joined = cv2.morphologyEx(grass, cv2.MORPH_CLOSE, k_pitch)

    num, labels, stats, centroids = cv2.connectedComponentsWithStats(joined, connectivity=8)
    if num <= 1:
        return grass, np.zeros_like(grass), np.zeros_like(grass), []
    biggest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    pitch = (labels == biggest).astype(np.uint8) * 255

    contours, _ = cv2.findContours(pitch, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(max(contours, key=cv2.contourArea))
    field = np.zeros_like(grass)
    cv2.fillConvexPoly(field, hull, 255)

    #Step 3: non-grass inside the field, then opening (lines) and closing (join pieces)
    non_grass = cv2.bitwise_and(field, cv2.bitwise_not(grass))
    k_open_people = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
    k_close_people = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_people, close_people))
    non_grass = cv2.morphologyEx(non_grass, cv2.MORPH_OPEN, k_open_people)
    non_grass = cv2.morphologyEx(non_grass, cv2.MORPH_CLOSE, k_close_people)

    #Step 4: blobs -> boxes that look like a person
    num, labels, stats, centroids = cv2.connectedComponentsWithStats(non_grass, connectivity=8)
    img_h, img_w = non_grass.shape
    boxes = []
    for i in range(1, num):
        x = int(stats[i, cv2.CC_STAT_LEFT])
        y = int(stats[i, cv2.CC_STAT_TOP])
        w = int(stats[i, cv2.CC_STAT_WIDTH])
        h = int(stats[i, cv2.CC_STAT_HEIGHT])
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area or h < min_height:
            continue
        if area > max_area_fraction * img_h * img_w:
            continue
        if h / w < min_aspect or h / w > max_aspect:
            continue
        boxes.append((x, y, w, h))
    return grass, field, non_grass, boxes

#IoU of two boxes (x, y, w, h): overlap area divided by the area they cover together
def iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1 = max(ax, bx)
    y1 = max(ay, by)
    x2 = min(ax + aw, bx + bw)
    y2 = min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = aw * ah + bw * bh - inter
    if union <= 0:
        return 0
    return inter / union

#For one frame: was each ground-truth person found (hits)? and how many of our boxes are real people (good_pred)?
def evaluate_frame(pred_boxes, gt_people):
    hits = []
    for g in gt_people:
        gbox = (g["x"], g["y"], g["w"], g["h"])
        best = 0
        for pbox in pred_boxes:
            best = max(best, iou(gbox, pbox))
        hits.append(best >= 0.3)

    good_pred = 0
    for pbox in pred_boxes:
        best = 0
        for g in gt_people:
            best = max(best, iou((g["x"], g["y"], g["w"], g["h"]), pbox))
        if best >= 0.3:
            good_pred += 1
    return hits, good_pred

In [ ]:
#Running the pipeline on every frame and measuring it
total_gt = 0
total_hit = 0
total_pred = 0
total_good = 0
gt_heights = []        #height of each ground-truth person
gt_hits = []           #was that person found?

for i, f in enumerate(frames):
    grass, field, non_grass, pred_boxes = find_people(f["image"], hue_low, hue_high, sat_min, open_people)
    hits, good_pred = evaluate_frame(pred_boxes, f["people"])

    #Saving what we need later inside the frame
    f["grass"] = grass
    f["pred"] = pred_boxes
    f["n_gt"] = len(f["people"])

    #Recall, precision and F1 (a single number that combines both) of this frame
    rec = 0
    if len(hits) > 0:
        rec = sum(hits) / len(hits)
    prec = 0
    if len(pred_boxes) > 0:
        prec = good_pred / len(pred_boxes)
    f["f1"] = 0
    if rec + prec > 0:
        f["f1"] = 2 * rec * prec / (rec + prec)

    total_gt += len(f["people"])
    total_hit += sum(hits)
    total_pred += len(pred_boxes)
    total_good += good_pred
    for g in f["people"]:
        gt_heights.append(g["h"])
    gt_hits += hits

    if (i + 1) % 10 == 0:
        print("Processed", i + 1, "of", len(frames), "frames")

print()
print("People in the ground truth:", total_gt, ", boxes we drew:", total_pred)
print("Recall (people found):", round(total_hit / total_gt, 3))
print("Precision (our boxes that are people):", round(total_good / total_pred, 3))

## [PRESENTATION] Does the size of the player matter?

The proposal said we would look at how accuracy changes with small players. Here is the recall for small, medium and large people (box height in pixels).

In [ ]:
gt_heights = np.array(gt_heights)
gt_hits = np.array(gt_hits)

#Height groups: smaller than 40 pixels, 40-80, bigger than 80
small = gt_heights < 40
medium = (gt_heights >= 40) & (gt_heights < 80)
large = gt_heights >= 80

recall_small = gt_hits[small].mean() if small.sum() > 0 else 0
recall_medium = gt_hits[medium].mean() if medium.sum() > 0 else 0
recall_large = gt_hits[large].mean() if large.sum() > 0 else 0

print("Small (< 40 px):", int(small.sum()), "people, recall", round(float(recall_small), 3))
print("Medium (40-80 px):", int(medium.sum()), "people, recall", round(float(recall_medium), 3))
print("Large (> 80 px):", int(large.sum()), "people, recall", round(float(recall_large), 3))

plt.figure(figsize=(7, 4))
plt.bar(["small (< 40 px)", "medium (40-80 px)", "large (> 80 px)"], [recall_small, recall_medium, recall_large])
plt.ylim(0, 1)
plt.title("Detection recall by player size")
plt.ylabel("Recall")
plt.tight_layout()
plt.show()

## [PRESENTATION] Hyperparameters of the detection

Which settings affect the result? We change **one setting at a time** (the first line is the default) and measure recall and precision. To save time, these tests use every second frame.

> **[ASSESSMENT] Hyperparameter effect (you should be able to say the *trends*, not the numbers).**
> - *Hue range:* too narrow loses grass in sun or shadow, which makes false "people" shapes; too wide includes other things (like green kits) in the grass.
> - *Minimum saturation:* a low value lets grey and white regions count as grass, so pitch lines can disappear into the grass; a high value drops pale grass.
> - *Opening size for the people:* too small leaves the pitch lines (many false boxes, lower precision); too big erases thin or small players (lower recall).

In [ ]:
#Frames used for the tests (every second frame, to save time)
exp_frames = frames[::2]

#Each test is: name, hue low, hue high, minimum saturation, opening size for the people
tests = [
    ("default", 35, 85, 40, 7),
    ("hue 30-90", 30, 90, 40, 7),
    ("hue 40-80", 40, 80, 40, 7),
    ("hue 45-75", 45, 75, 40, 7),
    ("min saturation 20", 35, 85, 20, 7),
    ("min saturation 60", 35, 85, 60, 7),
    ("min saturation 80", 35, 85, 80, 7),
    ("opening size 3", 35, 85, 40, 3),
    ("opening size 5", 35, 85, 40, 5),
    ("opening size 9", 35, 85, 40, 9),
    ("opening size 11", 35, 85, 40, 11),
]

for name, h_low, h_high, s_min, k_open in tests:
    gt_total = 0
    hit_total = 0
    pred_total = 0
    good_total = 0

    for f in exp_frames:
        grass, field, non_grass, pred_boxes = find_people(f["image"], h_low, h_high, s_min, k_open)
        hits, good_pred = evaluate_frame(pred_boxes, f["people"])
        gt_total += len(hits)
        hit_total += sum(hits)
        pred_total += len(pred_boxes)
        good_total += good_pred

    recall = hit_total / gt_total
    precision = good_total / pred_total if pred_total > 0 else 0
    print(name, ": recall", round(recall, 3), ", precision", round(precision, 3))

# PART 5 - Grouping People into Teams, Referees and Goalkeepers

Each person is now described by one color (the dominant torso color, three numbers B, G, R). We group those colors with **K-means**.

**K-means in plain words.**
1. Choose K and put K "centers" (starting colors) somewhere in color space.
2. Give every person to the nearest center.
3. Move each center to the average color of the people it received.
4. Repeat 2 and 3 until nothing changes.

It is *unsupervised*: it never sees the labels, it only sees colors. Two teams in different colors, a referee in a third color and goalkeepers in yet another color should appear as separate groups. A good choice is **K = 5**: team A, team B, referees, goalkeeper A, goalkeeper B (the same five groups used in the paper of my literature review).

**Two ways to cluster.** Our frames are meant to be independent stills, but a match has fixed kits, so we compare:
- **per image:** cluster the people of one frame only (truly independent stills, but few people per frame),
- **per clip:** cluster all the frames of the same clip together (more data, as if the kits of the match were learned from several photos of it).

**How we measure.**
- **Team accuracy** (players only, K = 2): the clusters are anonymous (cluster 0 is not "left"), so we try both ways of naming the two clusters and keep the better one.
- **Purity** (everyone, K = 5): in each cluster, take the most common true group (left player, right player, goalkeeper, referee...). Purity is the fraction of people who belong to the majority group of their cluster. 1.0 = every cluster contains only one kind of person.

**Two experiments.** First with the **ground-truth boxes** (this shows how good the color grouping is on its own, an upper bound), then with **our detected boxes** (the real, complete pipeline).

## Step 5 again as a function
`shirt_colours` repeats Step 5 of Part 3 (torso crop, remove grass, dominant color) and also computes the plain average color, so we can compare both.

In [ ]:
#Shirt color of one person: returns (dominant color, average color), both in B, G, R. Returns (None, None) if there are too few pixels
def shirt_colours(image_bgr, grass, box):
    img_h, img_w = grass.shape
    x, y, w, h = box

    #Torso crop (15%-50% of the height, middle 60% of the width), kept inside the image
    tx1 = max(0, min(int(x + 0.2 * w), img_w))
    tx2 = max(0, min(int(x + 0.8 * w), img_w))
    ty1 = max(0, min(int(y + 0.15 * h), img_h))
    ty2 = max(0, min(int(y + 0.5 * h), img_h))
    if tx2 - tx1 < 2 or ty2 - ty1 < 2:
        return None, None

    crop = image_bgr[ty1:ty2, tx1:tx2]
    crop_grass = grass[ty1:ty2, tx1:tx2]

    #Only the pixels that are not grass
    pixels = crop[crop_grass == 0]
    if len(pixels) < 10:
        return None, None

    #Average color
    mean_colour = pixels.mean(axis=0)

    #Dominant color (most common quantized color)
    q = pixels.astype(int) // 32
    ids = q[:, 0] * 64 + q[:, 1] * 8 + q[:, 2]
    best = int(np.argmax(np.bincount(ids, minlength=512)))
    mode_colour = np.array([(best // 64) * 32 + 16, ((best // 8) % 8) * 32 + 16, (best % 8) * 32 + 16], dtype=float)

    return mode_colour, mean_colour

#K-means on the colors, separately for each group (a group is a clip, or a single image).
#colours = one color per person (a table with 3 columns), group_ids = the group number of each person.
#Returns the cluster number of each person (-1 if its group has fewer than K people) and the center color of its cluster
def cluster_groups(colours, group_ids, K):
    clusters = np.full(len(colours), -1)
    centres = np.zeros((len(colours), 3))

    #Stop after 50 rounds or when the centers move less than 0.5
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 0.5)

    for g in np.unique(group_ids):
        idx = np.where(group_ids == g)[0]          #positions of the people of this group
        if len(idx) < K:
            continue

        data = np.float32(colours[idx])
        cv2.setRNGSeed(481)                        #fixed seed, so the results are the same every run
        #cv2.kmeans(data, K, no initial labels, stop rule, 10 attempts (keeps the best), smart starting centers)
        compactness, labels, centers = cv2.kmeans(data, K, None, criteria, 10, cv2.KMEANS_PP_CENTERS)
        labels = labels.flatten()
        clusters[idx] = labels
        centres[idx] = centers[labels]
    return clusters, centres

#Team accuracy for players clustered with K = 2. Only groups where both teams are present are counted
def team_accuracy(clusters, labels, group_ids):
    correct = 0
    total = 0
    groups_used = 0
    for g in np.unique(group_ids):
        sel = (group_ids == g) & (clusters >= 0)
        lab = labels[sel]
        cl = clusters[sel]
        if np.sum(lab == "player_left") == 0 or np.sum(lab == "player_right") == 0:
            continue
        #People for whom "cluster 0 = left team" is right. The other naming gives the opposite count
        same = int(np.sum((cl == 0) == (lab == "player_left")))
        correct += max(same, len(lab) - same)
        total += len(lab)
        groups_used += 1
    if total == 0:
        return 0, 0
    return correct / total, groups_used

#Purity of the clusters: fraction of people who have the most common true group of their cluster ("none" boxes are ignored)
def purity(clusters, labels, group_ids):
    good = 0
    total = 0
    for g in np.unique(group_ids):
        for c in np.unique(clusters[group_ids == g]):
            if c < 0:
                continue
            sel = (group_ids == g) & (clusters == c) & (labels != "none")
            if sel.sum() == 0:
                continue
            values, counts = np.unique(labels[sel], return_counts=True)
            good += counts.max()
            total += sel.sum()
    if total == 0:
        return 0
    return good / total

## Building the table of people

One row per person: its shirt colors (dominant and average), its clip number, its frame number and its true group. Two tables: one for the **ground-truth boxes** and one for **our detected boxes**. For a detected box, the true group is the group of the ground-truth person it matches (IoU of at least 0.3), or `"none"` if it matches nobody (a false alarm).

In [ ]:
#Table 1: people from the ground-truth boxes
gt_mode = []          #dominant shirt color
gt_mean = []          #average shirt color
gt_clip = []          #clip number
gt_frame = []         #frame number
gt_label = []         #true group
gt_is_player = []     #is it a player? (True / False)

for i, f in enumerate(frames):
    for g in f["people"]:
        box = (int(g["x"]), int(g["y"]), int(g["w"]), int(g["h"]))
        mode_c, mean_c = shirt_colours(f["image"], f["grass"], box)
        if mode_c is None:
            continue
        gt_mode.append(mode_c)
        gt_mean.append(mean_c)
        gt_clip.append(f["clip_id"])
        gt_frame.append(i)
        gt_label.append(g["label"])
        gt_is_player.append(g["role"] == "player")

#Lists to arrays, so we can select people with True / False masks
gt_mode = np.array(gt_mode)
gt_mean = np.array(gt_mean)
gt_clip = np.array(gt_clip)
gt_frame = np.array(gt_frame)
gt_label = np.array(gt_label)
gt_is_player = np.array(gt_is_player)
print("People in the ground-truth table:", len(gt_label))

#Table 2: people from OUR detected boxes
det_mode = []
det_clip = []
det_frame = []
det_label = []
det_box = []
det_is_player = []

for i, f in enumerate(frames):
    for pbox in f["pred"]:
        #Which ground-truth person does this box match best?
        best = 0
        best_label = "none"
        for g in f["people"]:
            v = iou((g["x"], g["y"], g["w"], g["h"]), pbox)
            if v > best:
                best = v
                best_label = g["label"]
        if best < 0.3:
            best_label = "none"

        mode_c, mean_c = shirt_colours(f["image"], f["grass"], pbox)
        if mode_c is None:
            continue
        det_mode.append(mode_c)
        det_clip.append(f["clip_id"])
        det_frame.append(i)
        det_label.append(best_label)
        det_box.append(pbox)
        det_is_player.append(best_label.startswith("player_"))

det_mode = np.array(det_mode)
det_clip = np.array(det_clip)
det_frame = np.array(det_frame)
det_label = np.array(det_label)
det_is_player = np.array(det_is_player)

print("Detected boxes with a color:", len(det_label), ", matched to a real person:", int(np.sum(det_label != "none")))

## Experiment 5A - Team accuracy with ground-truth boxes (color grouping on its own)

Players only, K = 2. We compare the two shirt-color descriptors (dominant color vs average color) and the two clustering scopes (per clip vs per image).

In [ ]:
print("TEAM ACCURACY (ground-truth boxes, players only, K = 2)")

#Players only: gt_is_player is a True / False mask
labels_p = gt_label[gt_is_player]

for color_name, colours in [("dominant color", gt_mode), ("average color", gt_mean)]:
    colours_p = colours[gt_is_player]

    #Clustering each clip separately
    clusters, centres = cluster_groups(colours_p, gt_clip[gt_is_player], 2)
    acc, used = team_accuracy(clusters, labels_p, gt_clip[gt_is_player])
    print("  ", color_name, ", cluster per clip: accuracy", round(acc, 3), "(groups evaluated:", used, ")")

    #Clustering each image separately
    clusters, centres = cluster_groups(colours_p, gt_frame[gt_is_player], 2)
    acc, used = team_accuracy(clusters, labels_p, gt_frame[gt_is_player])
    print("  ", color_name, ", cluster per image: accuracy", round(acc, 3), "(groups evaluated:", used, ")")

## Experiment 5B - How many groups? Purity for different K (ground-truth boxes)

All people (players, goalkeepers, referees), clustered per clip. If K = 5 is the right choice, purity should be high at K = 5 and should not improve much when K grows.

In [ ]:
print("PURITY (ground-truth boxes, everyone, dominant color, cluster per clip)")
for K in [2, 3, 4, 5, 6]:
    clusters, centres = cluster_groups(gt_mode, gt_clip, K)
    print("  K =", K, ": purity", round(purity(clusters, gt_label, gt_clip), 3))

## Experiment 5C - The real pipeline: our detected boxes

Now the boxes come from our own detection (Part 4). Boxes that do not match any ground-truth person have the group `"none"` and are ignored when measuring, but they are still clustered (the pipeline does not know which boxes are wrong).

In [ ]:
#Team accuracy: cluster only the detected boxes that match a player, K = 2
labels_p = det_label[det_is_player]

clusters, centres = cluster_groups(det_mode[det_is_player], det_clip[det_is_player], 2)
acc, used = team_accuracy(clusters, labels_p, det_clip[det_is_player])
print("TEAM ACCURACY, detected boxes, cluster per clip :", round(acc, 3), "(groups evaluated:", used, ")")

clusters, centres = cluster_groups(det_mode[det_is_player], det_frame[det_is_player], 2)
acc, used = team_accuracy(clusters, labels_p, det_frame[det_is_player])
print("TEAM ACCURACY, detected boxes, cluster per image:", round(acc, 3), "(groups evaluated:", used, ")")

#Purity with K = 5 on all the detected boxes (these K = 5 clusters are also used for the figures of Part 6)
det_clusters, det_centres = cluster_groups(det_mode, det_clip, 5)
print("PURITY, detected boxes, K = 5, cluster per clip:", round(purity(det_clusters, det_label, det_clip), 3))

> **[ASSESSMENT] Evaluation questions.**
> *How do you evaluate?* Detection: recall and precision with IoU >= 0.3 against the dataset boxes. Teams: accuracy with K = 2 on players (trying both cluster namings) and purity with K = 5 on everyone. Qualitative: example frames with successes and failures.
> *Why two kinds of experiments?* The ground-truth boxes show how good the color grouping is when detection is perfect; the detected boxes show the full pipeline. The gap between them is the cost of detection errors.
> *What did you compare?* Dominant color vs average color; clustering per clip vs per image; different K; different grass and morphology settings.
> *Ballpark numbers?* Say them like this: "with method A the accuracy was around X%, with method B around Y%." Copy X and Y from the tables above.
> *Why K-means and not something else?* It is simple, covered in the course, needs no labels, and the benchmark paper's own baseline also uses it for teams.

# PART 6 - Success and Failure Examples

## [PRESENTATION] Best and worst frames

Frames are ranked by their detection F1 score. The first two below are the best frames and the last two are the worst. Our boxes are drawn in the **color of their K-means group center** (so a box is drawn in the color of the shirt group the pipeline assigned it to), and the ground truth is in thin white.

In [ ]:
#Frames with at least 5 labeled people, sorted by F1 (lowest first)
ranked = []
for i, f in enumerate(frames):
    if f["n_gt"] >= 5:
        ranked.append((f["f1"], i))
ranked.sort()

#Two best frames, then the two worst
gallery_ids = [ranked[-1][1], ranked[-2][1], ranked[0][1], ranked[1][1]]
gallery_titles = ["Success", "Success", "Failure", "Failure"]

plt.figure(figsize=(15, 9))

for k in range(4):
    i = gallery_ids[k]
    drawn = cv2.cvtColor(frames[i]["image"], cv2.COLOR_BGR2RGB)

    #Ground truth (thin white)
    for g in frames[i]["boxes"]:
        cv2.rectangle(drawn, (int(g["x"]), int(g["y"])), (int(g["x"] + g["w"]), int(g["y"] + g["h"])), (255, 255, 255), 1)

    #Our boxes, in the color of their cluster center (grey if the person was not clustered)
    for j in range(len(det_frame)):
        if det_frame[j] != i:
            continue
        x, y, w, h = det_box[j]
        if det_clusters[j] < 0:
            colour = (128, 128, 128)
        else:
            colour = (int(det_centres[j][2]), int(det_centres[j][1]), int(det_centres[j][0]))      #BGR -> RGB
        cv2.rectangle(drawn, (x, y), (x + w, y + h), colour, 3)

    plt.subplot(2, 2, k + 1)
    plt.imshow(drawn)
    plt.title(gallery_titles[k] + " - " + frames[i]["name"] + " (F1 " + str(round(frames[i]["f1"], 2)) + ", " + str(len(frames[i]["pred"])) + " boxes, " + str(frames[i]["n_gt"]) + " people)", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Look at the Pipeline Stages of a Failure Frame

To understand *why* a frame failed, look at its intermediate masks. This is exactly the habit the assessment asks about ("what worked, what did not, and what you changed").

In [ ]:
#Intermediate masks of the worst frame
worst = gallery_ids[2]
grass_w, field_w, non_grass_w, boxes_w = find_people(frames[worst]["image"], hue_low, hue_high, sat_min, open_people)

plt.figure(figsize=(15, 8))

plt.subplot(2, 2, 1)
plt.imshow(cv2.cvtColor(frames[worst]["image"], cv2.COLOR_BGR2RGB))
plt.title("Worst frame: " + frames[worst]["name"])
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(grass_w, cmap="gray")
plt.title("Grass mask")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(field_w, cmap="gray")
plt.title("Field region")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(non_grass_w, cmap="gray")
plt.title("Non-grass after morphology")
plt.axis("off")

plt.tight_layout()
plt.show()

# PART 7 - Sources of Error, Conclusion and Study Guide

## [PRESENTATION] Main sources of error (write YOUR observations here)

The course asks for **your own** analysis, so after looking at your figures, fill this in with what you actually saw. These are the usual suspects to check for; keep the ones that appear in your failure frames and delete the others:

- **Small, far-away players:** removed by the size filter or too few pixels in the torso for a reliable color (see the recall by size in Part 4).
- **Overlapping players:** touching players form one blob, so one box covers two people and one color is shared.
- **Kits similar to the grass:** a green goalkeeper or referee shirt is classified as grass and disappears.
- **Shadows and lighting:** shadowed grass can fall outside the hue/value range and become a false "person".
- **Things inside the field that are not people:** the ball, advertising boards on the sideline, pitch markings.
- **Teams with similar kit colors**, or goalkeepers and referees in colors close to a team: K-means cannot separate them (see purity).
- **Wrong field region:** the convex hull is wrong if the visible pitch is not convex or green objects appear elsewhere.

**My observations (fill in):**
- The pipeline works well when ...
- It fails when ...
- The setting that mattered the most was ...

## [PRESENTATION] Preliminary conclusion

- *Main findings:* (summarize your detection recall and precision, team accuracy and purity numbers in one or two sentences)
- *What I learned:* (for example, how much of the work is choosing good color spaces and cleaning masks with morphology)
- *Ideas to improve:* adapt the grass thresholds per image (for example, from the most common hue in the image), split touching players (for example, with distance-transform based methods), use color distance in a perceptual color space (Lab) for K-means, and separate goalkeepers and referees with extra cues.

## Video checklist (4-5 minutes, your own voice)

1. Say your name and the project title (Part 0).
2. Problem and why it matters; show the input and desired output figure (Part 2).
3. Pipeline diagram and one example for each step (Part 0 and Part 3).
4. Say which libraries you used (the table at the top).
5. Dataset: clips, frames, people per frame, small players (Part 2).
6. Preliminary results: detection recall and precision, team accuracy and purity (Parts 4 and 5), how you will finish the evaluation and which settings you studied.
7. Show a success and a failure example and the main sources of error (Part 6).
8. Conclusion: findings, what you learned, ideas to improve (above).

## Study guide for the assessment (the things this notebook lets you say)

- **Pipeline from memory:** HSV grass mask, opening, closing to rejoin the pitch, biggest piece, convex hull, non-grass inside the field, opening and closing, connected components, size and shape filter, torso crop, dominant color, K-means.
- **Course concepts used:** color spaces (HSV), thresholding, morphology (opening, closing, structuring element, convex hull), connected components, color descriptors (dominant color), K-means clustering.
- **Which parts are library and which are ours:** the algorithms (HSV conversion, morphology, components, K-means) come from OpenCV; the design of the pipeline, the settings, the torso rule, the evaluation and the experiments are ours.
- **Dataset:** SoccerNet-GSR, a few clips with evenly spaced frames, ballpark sizes, the noise in the images.
- **Evaluation:** IoU, recall, precision, team accuracy, purity, and why we need both ground-truth boxes and detected boxes.
- **Hyperparameters:** the trends of the hue range, minimum saturation, opening size, K, color descriptor and clustering scope.
- **Timeline story:** first look at the data and the labels, then the grass mask, then morphology to clean it, then the people blobs, then colors and K-means, then evaluation and tuning.
- **No pretrained models anywhere.**

# PART 8 - Improvements (the "next steps" from our presentation)

Part 7 listed the main sources of error. This part fixes the three that belong to the field and team side. Every change is a **switch**, so Parts 3-6 above are unchanged and every table below compares "before" and "after" on the same 60 frames.

| | Problem seen in Part 6 | Idea | Course concept |
|---|---|---|---|
| 8A | Shadowed grass falls outside the HSV thresholds and becomes false "people" | Learn **this frame's** two grass colors (sunlit and shaded) and accept pixels close to either; **lightness counts less than color** | color spaces (LAB), K-means, color distance |
| 8B | Touching players form one blob, so one box covers two people | Cut wide blobs at the **valleys of the column sums** (projection profile) | connected components, projection profiles |
| 8C | Measuring 8B fairly | **One-to-one** matching: a box can only find one person | evaluation |
| 8D | Shirt colors compared in B, G, R (not how people see color) | Cluster shirts in **LAB**, where distance matches how different colors look; optionally weight lightness less | color spaces, K-means |

## 8A - A grass mask that survives shadows

**Why the HSV mask fails in shadows.** Look at the worst frame of Part 6 (the stadium roof throws a shadow over half of the pitch). Measured on that frame:

| | Hue | Saturation | Value |
|---|---|---|---|
| Sunlit grass | about 35 (exactly our `hue_low`) | about 100 | about 150 |
| Shaded grass | about 66 | about 65 | about 50 (close to our `val_min = 40`) |

The shaded part is lit only by the blue sky, so its hue moves towards blue and it gets much darker. The deepest shade drops below `val_min`, becomes "not grass", and the morphology turns it into big false blobs. The sunlit grass sits right on the lower hue limit, so it is speckled too.

**The idea: learn the grass colors of each frame.**
1. **Seeds:** the HSV mask with a slightly wider hue range (30-90), **eroded** so only pixels deep inside the grass are kept. These are surely grass.
2. **K-means with K = 2** on the LAB colors of the seeds. In a frame with a shadow the two centers become "sunlit grass" and "shaded grass"; without a shadow they are the two mowing stripes. Either way they describe this frame's grass.
3. A pixel is **grass if its color is close to one of the two centers**. The distance is the LAB distance, but the **lightness difference counts only half** (`light_weight = 0.5`): a shadow mostly changes lightness, while a shirt differs in color (a, b).
4. The new mask **only adds** grass: every pixel the HSV mask of Part 3 calls grass stays grass. So where there is no shadow nothing changes (we checked: replacing the mask instead of adding to it made pitch lines thicker, they survived the opening and glued players together).
5. The **field region still comes from the original HSV mask** (Part 3). Dark crowd areas can look like shaded grass, so the new mask is only used to decide what is grass *inside* the field.

> **[ASSESSMENT] Shadow-robust grass.**
> *Why LAB and not HSV?* In LAB, L is lightness and a, b are the color, and distances roughly match how different two colors look. A shadow changes mostly L, so giving L a smaller weight makes shaded grass close to sunlit grass.
> *Why per frame?* Every stadium, camera and time of day has a different grass color. Learning it from the frame itself (from pixels we are sure about) is cheaper and more robust than one fixed threshold for all frames.
> *Why K = 2?* One center for sun and one for shade. If there is no shadow, the two centers are just two grass shades (stripes), which does no harm.
> *What can still go wrong?* Green shirts are still "grass"; very deep shadow with almost no color left can still be missed.

In [ ]:
#Settings of the shadow-robust grass mask
seed_hue_low = 30          #slightly wider hue range for the seeds
seed_hue_high = 90
seed_erode = 7             #erosion size: seeds must be deep inside the grass
light_weight = 0.5         #how much the lightness (L) difference counts (1 = like the other channels)
grass_dist = 18            #maximum (weighted) LAB distance to a grass color

#Shadow-robust grass mask. Returns the mask (255 = grass) and the two grass colors (LAB) of this frame
def grass_mask_robust(image_bgr, l_weight=light_weight, max_dist=grass_dist):
    #1. Seeds: HSV threshold with a wider hue range, eroded so only sure grass is kept
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    seeds = ((hsv[:, :, 0] >= seed_hue_low) & (hsv[:, :, 0] <= seed_hue_high) & (hsv[:, :, 1] >= sat_min) & (hsv[:, :, 2] >= val_min)).astype(np.uint8) * 255
    k_seed = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (seed_erode, seed_erode))
    seeds = cv2.erode(seeds, k_seed)

    #A light blur first, so the grass texture does not make single pixels jump in color
    lab = cv2.cvtColor(cv2.GaussianBlur(image_bgr, (5, 5), 0), cv2.COLOR_BGR2LAB).astype(np.float32)
    seed_pixels = lab[seeds > 0]
    if len(seed_pixels) < 100:                    #almost no grass: keep the plain HSV result
        hsv_grass = (hsv[:, :, 0] >= hue_low) & (hsv[:, :, 0] <= hue_high) & (hsv[:, :, 1] >= sat_min) & (hsv[:, :, 2] >= val_min)
        return hsv_grass.astype(np.uint8) * 255, None

    #2. K-means with K = 2 on (at most about 20000) seed colors: sunlit grass and shaded grass
    step = max(1, len(seed_pixels) // 20000)
    data = np.float32(seed_pixels[::step])
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.5)
    cv2.setRNGSeed(481)
    compactness, labels, centers = cv2.kmeans(data, 2, None, criteria, 3, cv2.KMEANS_PP_CENTERS)

    #3. Weighted LAB distance of every pixel to the nearest grass color (L counts l_weight, a and b count 1)
    weights = np.array([l_weight, 1.0, 1.0], dtype=np.float32)
    dist0 = np.sqrt(np.sum(((lab - centers[0]) * weights) ** 2, axis=2))
    dist1 = np.sqrt(np.sum(((lab - centers[1]) * weights) ** 2, axis=2))
    nearest = np.minimum(dist0, dist1)

    #4. The new mask only ADDS grass: whatever the HSV mask of Part 3 calls grass stays grass
    hsv_grass = (hsv[:, :, 0] >= hue_low) & (hsv[:, :, 0] <= hue_high) & (hsv[:, :, 1] >= sat_min) & (hsv[:, :, 2] >= val_min)
    grass = ((nearest < max_dist) | hsv_grass | (seeds > 0)).astype(np.uint8) * 255
    return grass, centers

In [ ]:
#Before / after on the worst frame of Part 6 and on the three example images
check_ids = [gallery_ids[2], gallery_ids[3], demo_ids[0], demo_ids[1]]

plt.figure(figsize=(15, 4 * len(check_ids)))
for k in range(len(check_ids)):
    image = frames[check_ids[k]]["image"]

    #Original grass mask (Part 3, step 1)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    old_grass = ((hsv[:, :, 0] >= hue_low) & (hsv[:, :, 0] <= hue_high) & (hsv[:, :, 1] >= sat_min) & (hsv[:, :, 2] >= val_min)).astype(np.uint8) * 255

    #New grass mask
    new_grass, centers = grass_mask_robust(image)

    plt.subplot(len(check_ids), 3, 3 * k + 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(frames[check_ids[k]]["name"])
    plt.axis("off")

    plt.subplot(len(check_ids), 3, 3 * k + 2)
    plt.imshow(old_grass, cmap="gray")
    plt.title("Before: HSV grass mask")
    plt.axis("off")

    plt.subplot(len(check_ids), 3, 3 * k + 3)
    plt.imshow(new_grass, cmap="gray")
    title = "After: two grass colors, L weighted " + str(light_weight)
    if centers is not None:
        title += "\nL of the two grass colors: " + str(int(centers[0][0])) + " and " + str(int(centers[1][0]))
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.savefig("fig_8a_grass_before_after.png", dpi=120)
plt.show()

## 8B - Splitting touching players

**The problem.** Two players who touch (or whose shadows touch) become **one connected component**, so one box covers both. Usually the merged blob is also too wide (height / width below 0.8), so the shape filter of Part 3 **throws it away** and both players are lost.

**The idea: vertical projection profile.** A standing person is about 0.45 times as wide as tall. If a blob is much wider than that, it probably holds several people:
1. Expected number of people: `n = round(width / (0.45 * height))`. If `n < 2`, nothing to do.
2. **Column sums** of the blob mask: for each column, how many white pixels it has. Between two people there is a **valley** (the gap between them, or only an arm or a shadow).
3. Only blobs that can be a group of people are split: at most 4 people wide, and solid (at least 30% of the box is white; a leftover pitch line or the center circle fills much less).
4. Look for the lowest column near each expected cut (`width / n`, `2 * width / n`, ...) and cut there, but only if it is a real valley (fewer than 60% of the pixels of the fullest column).
5. Each piece gets its own tight box, and the usual size and shape filter is applied to the pieces.

This is the same projection-profile idea the jersey notebook uses to cut two touching digits apart.

> **[ASSESSMENT] Splitting players.**
> *What is a projection profile?* The sum of a binary mask along columns (or rows). Objects give peaks and gaps between objects give valleys.
> *Why not always cut in the middle?* Two players are rarely the same width; the valley is where the gap really is.
> *Limits?* It only separates people standing side by side. One player in front of another (overlap in depth) has no valley; a distance-transform/watershed method would be the next step.

In [ ]:
#Settings of the splitting
person_aspect = 0.45        #expected width / height of one standing person
valley_ratio = 0.6          #a cut is accepted only where the column has < 60% of the pixels of the fullest column
max_people = 4              #at most this many people in one blob
min_fill = 0.3              #a group of people fills at least 30% of its box (a pitch line fills much less)

#How wide are the people in the ground truth? (checks the 0.45 above)
ratios = []
for f in frames:
    for g in f["people"]:
        ratios.append(g["w"] / g["h"])
print("Width / height of the labeled people: median", round(float(np.median(ratios)), 2),
      ", 10th-90th percentile", np.round(np.percentile(ratios, [10, 90]), 2))

#Split one blob (blob_mask = True / False array of the blob inside its box, x and y = top-left corner of the box)
#Returns a list of boxes (x, y, w, h): one box if nothing is split
def split_blob(blob_mask, x, y):
    h, w = blob_mask.shape
    #Only blobs that can be a group of people: solid enough (a leftover pitch line or arc fills little of its box)
    #and not wider than max_people people
    fill = blob_mask.sum() / (w * h)
    n = int(round(w / (person_aspect * h)))
    if n < 2 or n > max_people or fill < min_fill:
        return [(x, y, w, h)]

    #Column sums, smoothed a little so one noisy column does not decide the cut
    cols = blob_mask.sum(axis=0).astype(float)
    cols = np.convolve(cols, np.ones(5) / 5, mode="same")

    cuts = [0]
    for k in range(1, n):
        centre = int(k * w / n)                    #where the cut would be if all people had the same width
        half = max(1, int(0.3 * w / n))            #search around it
        lo = max(1, centre - half)
        hi = min(w - 1, centre + half)
        if hi <= lo:
            continue
        cut = lo + int(np.argmin(cols[lo:hi]))
        if cols[cut] < valley_ratio * cols.max():  #only real valleys
            cuts.append(cut)
    cuts.append(w)

    boxes = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        piece = blob_mask[:, a:b]
        rows = np.where(piece.any(axis=1))[0]
        cs = np.where(piece.any(axis=0))[0]
        if len(rows) == 0 or len(cs) == 0:
            continue
        #Tight box of the piece, back in image coordinates
        boxes.append((x + a + int(cs[0]), y + int(rows[0]), int(cs[-1] - cs[0] + 1), int(rows[-1] - rows[0] + 1)))
    return boxes

## 8C - The improved detection, and a fairer way to measure it

`find_people_v2` is `find_people` from Part 4 with two switches:
- `grass_mode = "hsv"` (Part 3) or `"robust"` (8A),
- `split = False / True` (8B).

**A fairer score.** In Part 4 a person counts as found if *any* box overlaps it with IoU of at least 0.3. A box that covers **two** side-by-side players overlaps each of them with IoU of about 0.5, so both are "found" by one box. That hides exactly the error 8B fixes. So we also report a **one-to-one** score: boxes and people are paired greedily from the highest IoU down, and each box can be paired with only one person (this is how detection benchmarks score).

In [ ]:
#find_people of Part 4 with two switches: the grass mask ("hsv" or "robust") and splitting touching players
def find_people_v2(image_bgr, grass_mode="hsv", split=False, l_weight=light_weight, max_dist=grass_dist):
    #Step 1: HSV grass mask, exactly as in Part 4
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    grass = ((hsv[:, :, 0] >= hue_low) & (hsv[:, :, 0] <= hue_high) & (hsv[:, :, 1] >= sat_min) & (hsv[:, :, 2] >= val_min)).astype(np.uint8) * 255
    k_grass = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_grass, open_grass))
    grass = cv2.morphologyEx(grass, cv2.MORPH_OPEN, k_grass)

    #Step 2: field region from the HSV mask (as in Part 4)
    k_pitch = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_pitch, close_pitch))
    joined = cv2.morphologyEx(grass, cv2.MORPH_CLOSE, k_pitch)
    num, labels, stats, centroids = cv2.connectedComponentsWithStats(joined, connectivity=8)
    if num <= 1:
        return grass, np.zeros_like(grass), np.zeros_like(grass), []
    biggest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    pitch = (labels == biggest).astype(np.uint8) * 255
    contours, _ = cv2.findContours(pitch, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(max(contours, key=cv2.contourArea))
    field = np.zeros_like(grass)
    cv2.fillConvexPoly(field, hull, 255)

    #8A: inside the field, the shadow-robust mask decides what is grass
    if grass_mode == "robust":
        grass, centers = grass_mask_robust(image_bgr, l_weight, max_dist)
        grass = cv2.morphologyEx(grass, cv2.MORPH_OPEN, k_grass)

    #Step 3: non-grass inside the field, opening (lines) and closing (pieces of a person)
    non_grass = cv2.bitwise_and(field, cv2.bitwise_not(grass))
    k_open_people = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_people, open_people))
    k_close_people = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_people, close_people))
    non_grass = cv2.morphologyEx(non_grass, cv2.MORPH_OPEN, k_open_people)
    non_grass = cv2.morphologyEx(non_grass, cv2.MORPH_CLOSE, k_close_people)

    #Step 4: blobs -> boxes (8B: wide blobs are split first, then every piece is filtered)
    num, labels, stats, centroids = cv2.connectedComponentsWithStats(non_grass, connectivity=8)
    img_h, img_w = non_grass.shape
    boxes = []
    for i in range(1, num):
        x = int(stats[i, cv2.CC_STAT_LEFT])
        y = int(stats[i, cv2.CC_STAT_TOP])
        w = int(stats[i, cv2.CC_STAT_WIDTH])
        h = int(stats[i, cv2.CC_STAT_HEIGHT])
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area or h < min_height:
            continue
        if area > max_area_fraction * img_h * img_w:
            continue

        if split:
            blob_mask = labels[y:y + h, x:x + w] == i
            pieces = split_blob(blob_mask, x, y)
        else:
            pieces = [(x, y, w, h)]

        for (px, py, pw, ph) in pieces:
            piece_area = int(np.sum(labels[py:py + ph, px:px + pw] == i))
            if piece_area < min_area or ph < min_height:
                continue
            if len(pieces) > 1 and piece_area / (pw * ph) < min_fill:     #a piece of a split must be solid too
                continue
            if ph / pw < min_aspect or ph / pw > max_aspect:
                continue
            boxes.append((px, py, pw, ph))
    return grass, field, non_grass, boxes

#One-to-one matching: pairs (person, box) sorted by IoU, highest first; each box and each person is used once
def evaluate_frame_1to1(pred_boxes, gt_people, thr=0.3):
    pairs = []
    for gi, g in enumerate(gt_people):
        for pi, pbox in enumerate(pred_boxes):
            v = iou((g["x"], g["y"], g["w"], g["h"]), pbox)
            if v >= thr:
                pairs.append((v, gi, pi))
    pairs.sort(reverse=True)
    used_gt = set()
    used_pred = set()
    for v, gi, pi in pairs:
        if gi in used_gt or pi in used_pred:
            continue
        used_gt.add(gi)
        used_pred.add(pi)
    hits = [gi in used_gt for gi in range(len(gt_people))]
    return hits, len(used_pred)

#Is a labeled person touching another labeled person? (their boxes overlap)
def touching_flags(gt_people):
    flags = []
    for a, g in enumerate(gt_people):
        t = False
        for b, o in enumerate(gt_people):
            if a != b and iou((g["x"], g["y"], g["w"], g["h"]), (o["x"], o["y"], o["w"], o["h"])) > 0:
                t = True
        flags.append(t)
    return flags

#Runs one version of the detection on a list of frames and returns its scores
def score_detection(frame_list, grass_mode, split, l_weight=light_weight, max_dist=grass_dist):
    gt_n = 0
    hit_any = 0
    hit_one = 0
    pred_n = 0
    good_any = 0
    good_one = 0
    touch_n = 0
    touch_hit = 0
    for f in frame_list:
        grass, field, non_grass, pred = find_people_v2(f["image"], grass_mode, split, l_weight, max_dist)
        hits, good = evaluate_frame(pred, f["people"])
        hits1, good1 = evaluate_frame_1to1(pred, f["people"])
        flags = touching_flags(f["people"])
        gt_n += len(hits)
        hit_any += sum(hits)
        hit_one += sum(hits1)
        pred_n += len(pred)
        good_any += good
        good_one += good1
        for t, h1 in zip(flags, hits1):
            if t:
                touch_n += 1
                touch_hit += h1
    return {"recall": hit_any / max(1, gt_n), "precision": good_any / max(1, pred_n),
            "recall_1to1": hit_one / max(1, gt_n), "precision_1to1": good_one / max(1, pred_n),
            "touching_recall_1to1": touch_hit / max(1, touch_n), "touching_people": touch_n, "boxes": pred_n}

### [PRESENTATION] Detection before and after (all 60 frames)

Each row turns on one change, the last row has both. "touching" = labeled people whose box overlaps another labeled person (the cases 8B is meant for).

In [ ]:
versions = [("Part 4 (HSV, no split)", "hsv", False),
            ("8A shadow-robust grass", "robust", False),
            ("8B split touching players", "hsv", True),
            ("8A + 8B", "robust", True)]

detection_results = {}
print("version                      recall  precision | 1-to-1: recall  precision  touching recall | boxes")
for name, mode, split in versions:
    r = score_detection(frames, mode, split)
    detection_results[name] = r
    print(name.ljust(28), str(round(r["recall"], 3)).rjust(6), str(round(r["precision"], 3)).rjust(10), "|",
          str(round(r["recall_1to1"], 3)).rjust(14), str(round(r["precision_1to1"], 3)).rjust(10),
          str(round(r["touching_recall_1to1"], 3)).rjust(16), "|", r["boxes"])
print("(touching people in the labels:", detection_results[versions[0][0]]["touching_people"], ")")

## 8D - Team colors in LAB

Part 5 describes each shirt by its dominant **B, G, R** color, and K-means measures distances in B, G, R. But equal steps in B, G, R do not look equally different to people, and a shadow on a shirt changes all three channels at once.

In **LAB**, L = lightness, a = green-red, b = blue-yellow, and the distance between two colors roughly matches how different they look. We compare:
- **B, G, R** dominant and average (Part 5),
- **LAB** dominant and average,
- **LAB with lightness weighted 0.5**: a shirt in the shade and the same shirt in the sun get closer,
- **only a, b**: lightness ignored completely (risky: a white and a black kit have the same a, b!).

The torso crop and the grass removal are exactly those of Part 5, so only the color space changes.

> **[ASSESSMENT] LAB for teams.**
> *Why should LAB help?* K-means uses Euclidean distance; in LAB that distance is close to the perceived color difference, so the clusters follow what a person would call "the same kit".
> *Why not drop lightness completely?* White and black kits differ almost only in lightness. Weighting it down keeps them apart while making shadows matter less.

In [ ]:
#Shirt color in LAB: returns (dominant LAB color, average LAB color). OpenCV LAB: L, a, b all in 0-255 (a, b: 128 = neutral)
def shirt_colours_lab(image_bgr, grass, box):
    img_h, img_w = grass.shape
    x, y, w, h = box
    #Same torso crop as Part 5
    tx1 = max(0, min(int(x + 0.2 * w), img_w))
    tx2 = max(0, min(int(x + 0.8 * w), img_w))
    ty1 = max(0, min(int(y + 0.15 * h), img_h))
    ty2 = max(0, min(int(y + 0.5 * h), img_h))
    if tx2 - tx1 < 2 or ty2 - ty1 < 2:
        return None, None
    crop = cv2.cvtColor(image_bgr[ty1:ty2, tx1:tx2], cv2.COLOR_BGR2LAB)
    pixels = crop[grass[ty1:ty2, tx1:tx2] == 0]
    if len(pixels) < 10:
        return None, None

    mean_colour = pixels.mean(axis=0)

    #Dominant color: 16 levels per channel (a and b only use a small part of 0-255, so 8 levels would be too coarse)
    q = pixels.astype(int) // 16
    ids = q[:, 0] * 256 + q[:, 1] * 16 + q[:, 2]
    best = int(np.argmax(np.bincount(ids, minlength=4096)))
    mode_colour = np.array([(best // 256) * 16 + 8, ((best // 16) % 16) * 16 + 8, (best % 16) * 16 + 8], dtype=float)
    return mode_colour, mean_colour

#LAB colors of the same ground-truth people as Table 1 of Part 5 (same order, same people skipped)
gt_lab_mode = []
gt_lab_mean = []
for i, f in enumerate(frames):
    for g in f["people"]:
        box = (int(g["x"]), int(g["y"]), int(g["w"]), int(g["h"]))
        mode_c, mean_c = shirt_colours(f["image"], f["grass"], box)
        if mode_c is None:
            continue
        lab_mode, lab_mean = shirt_colours_lab(f["image"], f["grass"], box)
        gt_lab_mode.append(lab_mode)
        gt_lab_mean.append(lab_mean)
gt_lab_mode = np.array(gt_lab_mode)
gt_lab_mean = np.array(gt_lab_mean)
print("LAB colors computed for", len(gt_lab_mode), "people (Part 5 table:", len(gt_label), ")")

### [PRESENTATION] Team accuracy (K = 2, players) and purity (K = 5, everyone), ground-truth boxes

In [ ]:
#Each descriptor: name, colors of every person. The weights multiply L (column 0); K-means then works on the weighted colors
descriptors = [("BGR dominant (Part 5)", gt_mode),
               ("BGR average (Part 5)", gt_mean),
               ("LAB dominant", gt_lab_mode),
               ("LAB average", gt_lab_mean),
               ("LAB average, L x 0.5", gt_lab_mean * np.array([0.5, 1, 1])),
               ("LAB average, only a b", gt_lab_mean * np.array([0.0, 1, 1]))]

team_results = {}
print("descriptor                 team acc. per clip   per image   purity K=5 per clip")
for name, colours in descriptors:
    cl, ce = cluster_groups(colours[gt_is_player], gt_clip[gt_is_player], 2)
    acc_clip, used = team_accuracy(cl, gt_label[gt_is_player], gt_clip[gt_is_player])
    cl, ce = cluster_groups(colours[gt_is_player], gt_frame[gt_is_player], 2)
    acc_img, used = team_accuracy(cl, gt_label[gt_is_player], gt_frame[gt_is_player])
    cl, ce = cluster_groups(colours, gt_clip, 5)
    pur = purity(cl, gt_label, gt_clip)
    team_results[name] = (acc_clip, acc_img, pur)
    print(name.ljust(26), str(round(acc_clip, 3)).rjust(12), str(round(acc_img, 3)).rjust(12), str(round(pur, 3)).rjust(14))

## [PRESENTATION] What Part 8 changed (fill in with your numbers)

- **Shadows (8A):** detection before → after: ...
- **Touching players (8B):** one-to-one recall on touching people: ... → ... (the old "any overlap" score could not see this error).
- **LAB team colors (8D):** team accuracy ... → ..., purity ... → ...
- **Still open:** players overlapping in depth (no valley to cut), green kits, players smaller than 40 px.

# Part II: Jersey numbers (Sidra)

# Seeing the Game: Jersey Number Recognition (classical pipeline)
**CSC 481 · Sidra Azeem** · team: Muzammil Ali Khan, Sidra Azeem

Reads a player's jersey number from a small still image with classical image processing, and answers
**"unreadable"** when it isn't confident.

| Step | What it does | Main image processing |
|---|---|---|
| 0 | Load data | – |
| 1 | Pick usable images | cropping, bicubic interpolation, Laplacian sharpness |
| 2 | Find the shirt | HSV/LAB colour spaces, K-means, colour distance, morphology, convex hull |
| 3 | Number mask | Otsu thresholding, median filter, opening/closing, border clearing |
| 4 | Split into digits | connected components, shape rules, projection profile |
| 5 | Read the digits | shape features (contours, Hu moments, zoning), HOG, SVM / kNN, small CNN from scratch |
| 6 | Decide + evaluate | confidence threshold, player-level vote |
| A1–A4 | Analyses | ablation, confusion matrix, pipeline figure, deblurring |

**How to run**
* **First time ever:** set `DOWNLOAD_DATA = True` in the setup cell, then Run all.
* **Every other time:** Run all. Slow results (synthetic digits, harvested digits, trained classifiers, test
  predictions) are saved to Drive and loaded instead of recomputed.
* CPU runtime is fine. A GPU only speeds up the optional CNN.

No pretrained models are used anywhere (course rule): every classifier is trained from scratch here.

## Step 0: Setup and data

In [ ]:
# ---- settings ----
DOWNLOAD_DATA = False      # True only the first time: downloads the zips to Drive
RUN_LABELLING = False     # True to label more images with the click widget (Step 1.3); off so Run all never waits
LABEL_SPLIT = 'dev'        # which images the widget shows: 'dev' (to train B3) or 'test'
SHOW_CHECKS = True         # False hides the big check grids (faster, shorter notebook)
N_SYNTH = 8000             # synthetic players for training digits (Step 5)

import os, glob, json, random, math, re, time, zipfile, pickle
from collections import Counter, defaultdict
import numpy as np
import cv2
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED)

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/csc481_soccer'      # saved results + zips live here
    LOCAL = '/content/jersey'                         # fast local copy of the images
else:
    WORK, LOCAL = './csc481_soccer', './jersey'
DRIVE_DATA = os.path.join(WORK, 'SoccerNet')
CACHE = os.path.join(WORK, 'cache')
FIGS = os.path.join(WORK, 'figures')
for d in (DRIVE_DATA, CACHE, FIGS):
    os.makedirs(d, exist_ok=True)

def cached(name, make):
    """Load a saved result from Drive, or compute it once and save it."""
    path = os.path.join(CACHE, name + '.pkl')
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    out = make()
    with open(path, 'wb') as f:
        pickle.dump(out, f)
    return out

### 0.1 Download (first time only)
The zips go to Drive and are **never unzipped there** (writing 700k small files to Drive is very slow).

In [ ]:
if DOWNLOAD_DATA:
    os.system('pip -q install SoccerNet')
    from SoccerNet.Downloader import SoccerNetDownloader
    SoccerNetDownloader(LocalDirectory=DRIVE_DATA).downloadDataTask(task='jersey-2023', split=['train', 'test'])
print(os.listdir(os.path.join(DRIVE_DATA, 'jersey-2023')) if os.path.exists(os.path.join(DRIVE_DATA, 'jersey-2023')) else 'no data yet')

### 0.2 Copy 6 frames per player to local disk
Each player (tracklet) has ~500 near-identical frames. We extract only 6 evenly spaced ones per player.

In [ ]:
FRAMES_PER_TRACKLET = 6

def extract_subset(zip_path, out_dir, k=FRAMES_PER_TRACKLET):
    """Extract the label file + k evenly spaced frames per player."""
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        by_t, keep = defaultdict(list), []
        for n in names:
            m = re.search(r'images/([^/]+)/[^/]+\.jpg$', n)
            if m:
                by_t[m.group(1)].append(n)
            elif n.endswith('_gt.json'):
                keep.append(n)
        for files in by_t.values():
            files.sort()
            idx = np.linspace(0, len(files) - 1, min(k, len(files))).round().astype(int)
            keep += [files[i] for i in idx]
        for n in keep:
            z.extract(n, out_dir)
    print(f'{os.path.basename(zip_path)}: {len(by_t)} players, extracted {len(keep)} of {len(names)} files')

for split in ['train', 'test']:
    if not os.path.exists(os.path.join(LOCAL, split)):
        extract_subset(os.path.join(DRIVE_DATA, 'jersey-2023', f'{split}.zip'), LOCAL)

### 0.3 Load players and labels
One folder = one player clip (tracklet); one label per folder; **-1 = no number visible**.

In [ ]:
def load_split(split):
    """Return {tracklet_id: [image paths]} and {tracklet_id: label}."""
    gts = glob.glob(os.path.join(LOCAL, '**', f'{split}_gt.json'), recursive=True)
    labels = {}
    if gts:
        with open(gts[0]) as f:
            labels = {str(k): int(v) for k, v in json.load(f).items()}
    img_dirs = [d for d in glob.glob(os.path.join(LOCAL, '**', 'images'), recursive=True)
                if f'{os.sep}{split}{os.sep}' in d + os.sep]
    tracklets = {}
    if img_dirs:
        for d in sorted(os.listdir(img_dirs[0])):
            paths = sorted(glob.glob(os.path.join(img_dirs[0], d, '*.jpg')))
            if paths:
                tracklets[d] = paths
    return tracklets, labels

def sample_frames(paths, k=FRAMES_PER_TRACKLET):
    if len(paths) <= k:
        return list(paths)
    return [paths[i] for i in np.linspace(0, len(paths) - 1, k).round().astype(int)]

train_tracklets, train_labels = load_split('train')
test_tracklets, test_labels = load_split('test')
print(f'train: {len(train_tracklets)} players | test: {len(test_tracklets)} players')

### 0.4 Split train players into dev / val
* **dev** (80% of train players): build and train
* **val** (20%): choose settings such as the threshold τ
* **test**: final score only

The split is **by player**, so the same player never appears in two sets.

In [ ]:
players = sorted(t for t in train_tracklets if t in train_labels)
random.Random(SEED).shuffle(players)
val_ids = set(players[:int(0.2 * len(players))])

items = [dict(path=p, gt=train_labels[t], tid=t, split='val' if t in val_ids else 'dev')
         for t in players for p in sample_frames(train_tracklets[t])]
test_items = [dict(path=p, gt=test_labels[t], tid=t, split='test')
              for t in sorted(test_tracklets) if t in test_labels for p in sample_frames(test_tracklets[t])]
print(len(items), 'train stills', Counter(it['split'] for it in items), '|', len(test_items), 'test stills')

## Step 1: Pick usable images

### 1.1 Torso crop and upscaling
* **Torso crop**: drop the top 1/6 (head) and bottom 1/3 (shorts, legs), trim 8% per side (ratios from Grad 2025).
* **Upscale** to 128 px tall with **bicubic interpolation**, so every later rule works at the same scale.

In [ ]:
TORSO_H = 128

def torso_crop(img, top=1/6, bottom=1/3, side=0.08):
    h, w = img.shape[:2]
    y0, y1 = int(round(h * top)), int(round(h * (1 - bottom)))
    x0, x1 = int(round(w * side)), int(round(w * (1 - side)))
    return img[y0:max(y1, y0 + 1), x0:max(x1, x0 + 1)]

def upscale(img, target_h=TORSO_H):
    s = target_h / img.shape[0]
    return cv2.resize(img, (max(1, int(round(img.shape[1] * s))), target_h), interpolation=cv2.INTER_CUBIC)

def get_torso(path):
    return upscale(torso_crop(cv2.imread(path)))

def show_grid(imgs, titles=None, cols=8, size=1.6):
    rows = math.ceil(len(imgs) / cols)
    plt.figure(figsize=(cols * size, rows * size * 1.5))
    for i, im in enumerate(imgs):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im.ndim == 3 else im, cmap='gray')
        if titles is not None:
            plt.title(str(titles[i]), fontsize=8)
        plt.axis('off')
    plt.tight_layout(); plt.show()

ex = random.Random(1).sample([it for it in items if it['gt'] >= 0], 8)
show_grid([cv2.imread(it['path']) for it in ex] + [get_torso(it['path']) for it in ex], [it['gt'] for it in ex] * 2)

### 1.2 Image quality: size and sharpness
**Sharpness = variance of the Laplacian.** The Laplacian is an edge filter: sharp images have strong edges
(high variance), blurry images weak ones (low variance).

In [ ]:
MIN_H = 40     # players shorter than this are unreadable mush

def sharpness(img_bgr):
    return float(cv2.Laplacian(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var())

def add_quality(items_):
    """Adds height, sharpness and the size filter to every still."""
    for it in items_:
        img = cv2.imread(it['path'])
        it['h'] = img.shape[0]
        it['sharp'] = sharpness(torso_crop(img)) if img.shape[0] >= 8 else 0.0
        it['usable'] = it['h'] >= MIN_H

add_quality(items); add_quality(test_items)
H = np.array([it['h'] for it in items])
print(f"usable (>= {MIN_H}px): {sum(it['usable'] for it in items)}/{len(items)} "
      f"| height percentiles 10/50/90: {np.percentile(H, [10, 50, 90]).round()}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(H, 50); ax[0].set_title('crop height (px)')
ax[1].hist(np.log10(np.array([it['sharp'] for it in items]) + 1), 50); ax[1].set_title('log10 sharpness')
plt.show()

### 1.3 Hand-labelled "readable" set
Labels are per player, so many frames don't show the number. We hand-checked stills: **can a human read the
number?** The readable ones are our fair per-image test set. Set `RUN_LABELLING = True` to label more
(click the readable ones, then "Save & next grid").

In [ ]:
LEGIBLE_FILE = os.path.join(WORK, 'legible_labels.json')            # evaluation labels (readable set, test)
B3_FILE = os.path.join(WORK, 'legible_labels_b3.json')               # extra dev labels, ONLY to train B3
legible = json.load(open(LEGIBLE_FILE)) if os.path.exists(LEGIBLE_FILE) else {}
legible_b3 = json.load(open(B3_FILE)) if os.path.exists(B3_FILE) else {}

if RUN_LABELLING:
    import ipywidgets as W
    from IPython.display import display, clear_output
    pool_src = [it for it in items if it['split'] == 'dev'] if LABEL_SPLIT == 'dev' else test_items
    label_pool = [it for it in pool_src if it['gt'] >= 0 and it['usable']]
    random.Random(SEED).shuffle(label_pool)
    state, status, out = {}, W.Label(), W.Output()
    target, target_file = (legible_b3, B3_FILE) if LABEL_SPLIT == 'dev' else (legible, LEGIBLE_FILE)

    def render():
        todo = [it for it in label_pool if it['path'] not in legible and it['path'] not in legible_b3]
        status.value = f'{len(target)} labelled in this file, {sum(target.values())} readable'
        if not todo:
            return
        state['batch'] = todo[:24]
        state['toggles'], tiles = [], []
        for it in state['batch']:
            img = W.Image(value=cv2.imencode('.png', get_torso(it['path']))[1].tobytes(), format='png',
                          layout=W.Layout(width='100px', height='128px', object_fit='contain'))
            tb = W.ToggleButton(value=False, description=f"#{it['gt']}", layout=W.Layout(width='100px'))
            tb.observe(lambda ch, b=tb: setattr(b, 'button_style', 'success' if ch['new'] else ''), names='value')
            state['toggles'].append(tb)
            tiles.append(W.VBox([img, tb], layout=W.Layout(align_items='center')))
        with out:
            clear_output(wait=True)
            display(W.GridBox(tiles, layout=W.Layout(grid_template_columns='repeat(8, 110px)', grid_gap='6px')))

    def save_and_next(_):
        for it, tb in zip(state['batch'], state['toggles']):
            target[it['path']] = tb.value
        json.dump(target, open(target_file, 'w'))
        render()

    btn = W.Button(description='Save & next grid', button_style='primary', layout=W.Layout(width='180px'))
    btn.on_click(save_and_next)
    display(W.HBox([btn, status]), out)
    render()

In [ ]:
# The image sets used from here on
labelled = [it for it in items if it['path'] in legible]          # hand-checked train stills
readable = [it for it in labelled if legible[it['path']]]          # ... that a human can read
unread = [it for it in labelled if not legible[it['path']]]
test_labelled = [it for it in test_items if it['path'] in legible]
print(f'train: {len(labelled)} hand-labelled, {len(readable)} readable | test: {len(test_labelled)} hand-labelled')

if labelled:   # how readability depends on size
    hs = np.array([it['h'] for it in labelled]); ok = np.array([legible[it['path']] for it in labelled])
    edges = np.unique(np.percentile(hs, [0, 25, 50, 75, 100]))
    print('% readable by height:', ', '.join(f'{lo:.0f}-{hi:.0f}px: {ok[(hs >= lo) & (hs <= hi)].mean():.0%}'
                                             for lo, hi in zip(edges[:-1], edges[1:])))

## Pipeline switches
Every design choice that the **ablation study (A1)** turns off lives in `CFG`. Normal runs use the defaults.

In [ ]:
CFG_DEFAULT = dict(
    grass='colour',        # 'colour' = remove this image's grass colour | 'hsv' = generic green threshold
    search='silhouette',   # where to look for the number: 'silhouette' (player outline) | 'shirt' (shirt-colour hull)
    local_mask=True,       # also use the local-contrast masks (Step 3)
    strict=True,           # also use the stricter thresholds x1.3 and x1.6 (Step 3)
    clear_border=True,     # delete blobs touching left/right/bottom (Step 3)
    top_k=6,               # candidates passed from Step 4 to Step 5
    geo_mean=True,         # candidate confidence = geometric mean (else product) of digit probabilities
    one_penalty=0.7,       # multiplies the confidence of a lone '1' (1.0 = off)
    deblur=None,           # A4: None | 'unsharp' | 'wiener'
    chroma_mask=False,     # B1: also threshold colour-only contrast (LAB a,b) for pale numbers on white kits
    crops=((1/6, 1/3),),   # B2: torso windows (top, bottom) to try; the most confident reading wins
)
CFG = dict(CFG_DEFAULT)

class use_cfg:
    """with use_cfg(local_mask=False): ...  -> runs with changed switches, then restores them."""
    def __init__(self, **changes): self.changes = changes
    def __enter__(self): self.saved = dict(CFG); CFG.update(self.changes)
    def __exit__(self, *a): CFG.clear(); CFG.update(self.saved)

## Step 2: Find the shirt
The number is "pixels on the player whose colour differs strongly from the shirt". So we first find the shirt.

### 2.1 Grass removal
* **HSV** separates hue from brightness, so "green" is one threshold on hue and saturation.
* We then remove only pixels close (ΔE < 18) to **this image's grass colour**: a generic green threshold also
  deleted green shirts and lime numbers.

In [ ]:
from sklearn.cluster import KMeans

def grass_mask(bgr):
    """Generic grass: green hue with enough saturation (HSV threshold)."""
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, (33, 50, 30), (90, 255, 255)) > 0

def grass_colour_mask(lab, g, max_dE=18):
    """Pixels close to THIS image's grass colour (median LAB colour of the generic grass pixels)."""
    if g.sum() < 20:
        return np.zeros(g.shape, bool), None
    gcol = np.median(lab[g], axis=0)
    return np.linalg.norm(lab - gcol, axis=2) < max_dE, gcol

### 2.2 Shirt colour with K-means in LAB
In **LAB**, the distance between two colours roughly matches how different they look (Pan et al. 2024 found
it best for kit colours). **K-means (k = 3)** groups the torso pixels into 3 colours; the biggest is the shirt.

In [ ]:
def shirt_colour(lab, bad, k=3):
    """Biggest K-means cluster (in LAB) of the torso pixels that are not grass."""
    h = lab.shape[0]
    px = lab[h // 10: 9 * h // 10][~bad[h // 10: 9 * h // 10]]
    if len(px) < 30:
        return None
    km = KMeans(n_clusters=k, n_init=3, random_state=SEED).fit(px[::2])
    return km.cluster_centers_[np.bincount(km.labels_, minlength=k).argmax()]

### 2.3 Contrast map and search region
* **Contrast map ΔE** = LAB distance of each pixel to the shirt colour. Shirt is dark, the number lights up.
* **Region** = closing → largest connected component → convex hull. We search inside the **player silhouette**
  (everything that isn't grass): the tighter shirt-colour region missed numbers on striped kits.
* A **bilateral filter** first removes JPEG noise while keeping edges sharp.

In [ ]:
def hull_of_largest(m, close_frac=6):
    """Morphological closing -> largest connected component -> its convex hull."""
    h, w = m.shape
    ksz = max(3, w // close_frac) | 1
    closed = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksz, ksz)))
    n, lbl, st, _ = cv2.connectedComponentsWithStats(closed, 8)
    if n < 2:
        return np.zeros((h, w), bool)
    big = 1 + st[1:, cv2.CC_STAT_AREA].argmax()
    cnts, _ = cv2.findContours((lbl == big).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    r = np.zeros((h, w), np.uint8)
    cv2.drawContours(r, [cv2.convexHull(np.vstack(cnts))], -1, 1, -1)
    return r > 0

def shirt_analysis(torso_bgr, shirt_dE=25):
    """Step 2: shirt colour, contrast map dE, search region."""
    smooth = cv2.bilateralFilter(torso_bgr, 5, 40, 5)                   # denoise, keep edges
    lab = cv2.cvtColor(smooth, cv2.COLOR_BGR2LAB).astype(np.float32)
    g = grass_mask(smooth)
    grass, _ = grass_colour_mask(lab, g) if CFG['grass'] == 'colour' else (g, None)
    shirt = shirt_colour(lab, grass)
    if shirt is None:
        return None
    dE = np.linalg.norm(lab - shirt, axis=2)                            # number lights up
    shirt_px = (dE < shirt_dE) & ~grass
    region = hull_of_largest(~grass) if CFG['search'] == 'silhouette' else hull_of_largest(shirt_px)
    if not region.any():
        return None
    return dict(lab=lab, shirt=shirt, dE=dE, shirt_px=shirt_px, region=region, grass=grass)

def step2_panel(t, sa):
    """torso | contrast map | search region (cyan) with the shirt colour as a strip on top."""
    dE_vis = cv2.applyColorMap(np.clip(sa['dE'] * 2.5, 0, 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
    over = t.copy()
    cnts, _ = cv2.findContours(sa['region'].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(over, cnts, -1, (255, 255, 0), 2)
    over[:14] = cv2.cvtColor(np.uint8([[np.clip(sa['shirt'], 0, 255)]]), cv2.COLOR_LAB2BGR)[0, 0]
    return np.hstack([t, dE_vis, over])

def show_panels(panels, titles, cols=4, w=4, h=2.2):
    rows = math.ceil(len(panels) / cols)
    plt.figure(figsize=(cols * w, rows * h))
    for i, p in enumerate(panels):
        plt.subplot(rows, cols, i + 1); plt.imshow(p[..., ::-1]); plt.title(titles[i], fontsize=8); plt.axis('off')
    plt.tight_layout(); plt.show()

if SHOW_CHECKS and readable:
    ps = [(get_torso(it['path']), it) for it in readable[:16]]
    show_panels([step2_panel(t, shirt_analysis(t)) for t, _ in ps], [f"#{it['gt']}" for _, it in ps])

## Step 3: Number mask
* **Otsu** picks the threshold that best splits the pixels in two groups (number vs shirt), computed only
  inside the search region. A floor stops it from cutting into plain shirt noise.
* **Global** contrast = to the one shirt colour; **local** contrast = to the neighbourhood (a large **median
  filter** estimates the background), which helps on shaded white shirts.
* **Stricter thresholds** (x1.3, x1.6) separate a number that merged with an arm.
* **Opening** removes specks, **closing** reconnects broken strokes, **border clearing** deletes blobs touching
  the left/right/bottom edge (arms, other players, shorts).

In [ ]:
def otsu_on(values, floor):
    """Otsu threshold computed only on the given pixels, never below `floor`."""
    v = np.clip(values, 0, 255).astype(np.uint8).reshape(-1, 1)
    t, _ = cv2.threshold(v, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return max(float(t), floor)

def clean(m, open_k=2, close_k=3):
    m = m.astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((open_k, open_k), np.uint8))     # remove specks
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((close_k, close_k), np.uint8))  # reconnect strokes
    return m > 0

def clear_border_sides(m):
    """Delete blobs touching the left, right or bottom border (the top is allowed)."""
    n, lbl = cv2.connectedComponents(m.astype(np.uint8), connectivity=8)
    touch = set(np.unique(lbl[:, 0])) | set(np.unique(lbl[:, -1])) | set(np.unique(lbl[-1, :]))
    touch.discard(0)
    return m & ~np.isin(lbl, list(touch))

def local_contrast(lab, frac=0.5):
    """Distance of each pixel to its local background, estimated with a large median filter."""
    k = max(5, int(lab.shape[1] * frac)) | 1
    bg = np.dstack([cv2.medianBlur(np.clip(lab[..., i], 0, 255).astype(np.uint8), k) for i in range(3)])
    return np.linalg.norm(lab - bg.astype(np.float32), axis=2)

def number_masks(sa):
    """Step 3: candidate number masks (global/local contrast at 1-3 strictness levels)."""
    search = sa['region'] & ~sa['grass']
    if search.sum() < 30:
        return {}
    maps = [('global', sa['dE'], 30)]
    if CFG['local_mask']:
        maps.append(('local', local_contrast(sa['lab']), 15))
    if CFG['chroma_mask']:                                   # B1: colour difference only, brightness ignored
        maps.append(('chroma', np.linalg.norm(sa['lab'][..., 1:] - sa['shirt'][1:], axis=2), 12))
    factors = (1.0, 1.3, 1.6) if CFG['strict'] else (1.0,)
    out = {}
    for name, cmap, floor in maps:
        t0 = otsu_on(cmap[search], floor)
        for f in factors:
            m = clean((cmap > t0 * f) & search)
            out[name if f == 1.0 else f'{name}x{f}'] = clear_border_sides(m) if CFG['clear_border'] else m
    return out

if SHOW_CHECKS and readable:
    to3 = lambda m: cv2.cvtColor((m * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    panels = []
    for t, _ in ps:
        ms = number_masks(shirt_analysis(t))
        panels.append(np.hstack([t, to3(ms['global']), to3(ms.get('local', ms['global']))]))
    show_panels(panels, [f"#{it['gt']}  global | local" for _, it in ps])

## Step 4: Split the mask into digits
* **Connected components**: every separate blob. A digit must be 18–80% of the torso height, width/height
  0.1–1.4, fill ratio 0.2–0.9, and sit on the shirt (≥ 70% non-grass around it).
* **Pairs** = two blobs of similar height, side by side, close together.
* **Touching digits**: a too-wide blob is cut at the minimum of its **vertical projection profile**.
* The **top 6 candidates** go to Step 5, which picks the one it reads most confidently.

In [ ]:
def ring_fraction(ok_mask, x, y, w, h, pad=3):
    """Share of a thin ring around a box that is 'ok' (here: not grass)."""
    H, W = ok_mask.shape
    x0, y0, x1, y1 = max(0, x - pad), max(0, y - pad), min(W, x + w + pad), min(H, y + h + pad)
    ring = np.ones((y1 - y0, x1 - x0), bool)
    ring[y - y0:y - y0 + h, x - x0:x - x0 + w] = False
    return ok_mask[y0:y1, x0:x1][ring].mean() if ring.any() else 0.0

def digit_components(mask, sa):
    """Connected components shaped like a single digit."""
    H, W = mask.shape
    n, lbl, st, cents = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    comps = []
    for i in range(1, n):
        x, y, w, h, area = st[i]
        if not (0.18 * H <= h <= 0.8 * H): continue                    # digit height
        if w < 2 or not (0.1 <= w / h <= 1.4): continue                # thin '1' ... two touching digits
        if not (0.2 <= area / (w * h) <= 0.9): continue                # not a speck, not a solid block
        if ring_fraction(~sa['grass'], x, y, w, h) < 0.7: continue     # sits on the shirt
        cm = lbl[y:y + h, x:x + w] == i
        comps.append(dict(x=int(x), y=int(y), w=int(w), h=int(h), cx=float(cents[i][0]), mask=cm,
                          contrast=float(sa['dE'][y:y + h, x:x + w][cm].mean())))
    return comps

def split_wide(c):
    """Cut a wide blob at the minimum of its column sums (vertical projection profile)."""
    m = c['mask']; cols = m.sum(0).astype(float)
    a, b = int(.3 * len(cols)), int(.7 * len(cols))
    if b - a < 2:
        return [c]
    cut = a + int(np.argmin(cols[a:b])); parts = []
    for off, sub in ((0, m[:, :cut]), (cut, m[:, cut:])):
        ys, xs = np.where(sub.any(1))[0], np.where(sub.any(0))[0]
        if len(ys) == 0 or len(xs) == 0:
            return [c]
        parts.append(dict(c, x=c['x'] + off + int(xs[0]), y=c['y'] + int(ys[0]), w=len(xs), h=int(ys[-1] - ys[0] + 1),
                          cx=c['x'] + off + float(xs.mean()), mask=sub[ys[0]:ys[-1] + 1, xs[0]:xs[-1] + 1]))
    return parts

def group_score(g, W, H, split=False):
    """Higher = more number-like (tall, high contrast, centred, upper torso). None = not a valid pair."""
    hs = [c['h'] for c in g]
    cy = np.mean([c['y'] + c['h'] / 2 for c in g]) / H
    s = np.mean(hs) / H + np.mean([c['contrast'] for c in g]) / 150 - abs(np.mean([c['cx'] for c in g]) / W - .5)
    s -= 2 * max(0, cy - 0.6)                                          # numbers sit in the upper torso
    if len(g) == 2:
        a, b = sorted(g, key=lambda c: c['x'])
        if max(hs) / min(hs) > 1.35: return None                      # similar heights
        if min(a['y'] + a['h'], b['y'] + b['h']) - max(a['y'], b['y']) < .6 * min(hs): return None   # side by side
        gap = b['x'] - (a['x'] + a['w'])
        if gap > .6 * max(hs) or gap < -.2 * max(hs): return None       # close together
        if not split: s += .1                                           # bonus: two separate digits
    return s

def segment_digits(sa, masks):
    """Step 4: candidate groups of 1-2 digits from all masks, best first."""
    H, W = sa['dE'].shape
    cands = []
    for name, m in masks.items():
        base = digit_components(m, sa)
        groups = []                                     # (blobs, made_by_splitting)
        for i, a in enumerate(base):
            r = a['w'] / a['h']
            parts = split_wide(a) if r > .7 else [a]
            if r > .85 and len(parts) == 2:
                groups.append((parts, False))           # very wide: two touching digits
            else:
                groups.append(([a], False))
                if len(parts) == 2: groups.append((parts, True))
            for b in base[i + 1:]:
                groups.append(([a, b], False))
        for g, was_split in groups:
            s = group_score(g, W, H, split=was_split)
            if s is not None:
                cands.append(dict(score=float(s), source=name, boxes=sorted(g, key=lambda c: c['x'])))
    cands.sort(key=lambda c: -c['score'])
    return cands[:CFG['top_k']]

def draw_boxes(t, boxes, col=(0, 0, 255)):
    v = t.copy()
    for b in boxes:
        cv2.rectangle(v, (b['x'], b['y']), (b['x'] + b['w'] - 1, b['y'] + b['h'] - 1), col, 1)
    return v

### 4.1 Check Steps 2–4 on the readable images
Scored automatically: does the best candidate (or any of the top 6) have the right **number of digits**?

In [ ]:
def candidates_for(t):
    sa = shirt_analysis(t)
    return sa, (segment_digits(sa, number_masks(sa)) if sa else [])

if readable:
    top1 = topk = 0
    panels, titles = [], []
    for it in readable:
        t = get_torso(it['path']); _, cands = candidates_for(t)
        n = len(str(it['gt']))
        top1 += bool(cands) and len(cands[0]['boxes']) == n
        topk += any(len(c['boxes']) == n for c in cands)
        panels.append(np.hstack([draw_boxes(t, c['boxes']) for c in cands[:3]] + [t * 0] * (3 - len(cands[:3]))))
        titles.append(f"#{it['gt']}: " + ' | '.join(f"{len(c['boxes'])}d {c['source']}" for c in cands[:3]))
    print(f'right digit count: best candidate {top1}/{len(readable)} | among top {CFG["top_k"]}: {topk}/{len(readable)}')
    if SHOW_CHECKS:
        show_panels(panels[:16], titles[:16], h=2.3)

## Step 5: Read the digits

### 5.1 Training digits, part 1: realistic synthetic players
We draw a simple player (random kit, stripes, font, number) at 4x size, then shrink, blur, add noise and JPEG
compression so it looks like SoccerNet. Each one goes through **our own Steps 2–4**, and digits are kept only if
their boxes lie on the drawn number (so the labels are trustworthy).

In [ ]:
import matplotlib
from PIL import Image, ImageDraw, ImageFont

FONTS = [f for f in glob.glob('/usr/share/fonts/truetype/**/*Bold*.ttf', recursive=True)
         + [os.path.join(matplotlib.get_data_path(), 'fonts/ttf', n)
            for n in ('DejaVuSans-Bold.ttf', 'DejaVuSansMono-Bold.ttf', 'DejaVuSerif-Bold.ttf')]
         if os.path.exists(f)][:15]
PALETTE = [(255,255,255),(20,20,20),(200,30,30),(30,40,150),(250,220,0),(30,120,60),(240,120,20),
           (120,20,40),(90,180,230),(150,150,150),(200,255,60)]

def colour_dist(a, b):
    la, lb = (cv2.cvtColor(np.uint8([[c]]), cv2.COLOR_RGB2LAB)[0, 0].astype(float) for c in (a, b))
    return np.linalg.norm(la - lb)

def jitter(c, rng, a=25):
    return tuple(int(np.clip(v + rng.randint(-a, a), 0, 255)) for v in c)

def synth_player(number, rng, S=4):
    """Fake back-view player at real SoccerNet size. Returns (image, mask of the drawn number)."""
    W, H = rng.randint(30, 50), rng.randint(80, 130)
    Wb, Hb = W * S, H * S
    im = Image.new('RGB', (Wb, Hb), (rng.randint(40, 90), rng.randint(120, 170), rng.randint(30, 70)))   # grass
    d = ImageDraw.Draw(im)
    shirt = jitter(rng.choice(PALETTE), rng)
    numc = rng.choice([c for c in PALETTE if colour_dist(c, shirt) > 45])
    skin = jitter((200, 150, 120), rng, 40)
    cx = Wb // 2 + rng.randint(-Wb // 10, Wb // 10)
    tw = int(Wb * rng.uniform(.55, .8)); ty0, ty1 = int(Hb * .17), int(Hb * .6)
    d.ellipse((cx - Wb * .13, Hb * .02, cx + Wb * .13, Hb * .18), fill=(40, 30, 20))                # head
    for side in (-1, 1):                                                                          # arms
        xa = cx + side * tw // 2
        d.rectangle((min(xa, xa + side * Wb * .1), ty0 + 5, max(xa, xa + side * Wb * .1), ty1 - Hb * .1),
                    fill=skin if rng.random() < .6 else shirt)
    d.rounded_rectangle((cx - tw // 2, ty0, cx + tw // 2, ty1), radius=S * 3, fill=shirt)          # shirt
    if rng.random() < .25:                                                                        # stripes
        sc, horiz = jitter(rng.choice(PALETTE), rng), rng.random() < .5
        for p in range(0, Hb if horiz else Wb, S * 8):
            if horiz and ty0 + p < ty1: d.rectangle((cx - tw // 2, ty0 + p, cx + tw // 2, ty0 + p + S * 3), fill=sc)
            if not horiz and cx - tw // 2 + p < cx + tw // 2: d.rectangle((cx - tw // 2 + p, ty0, cx - tw // 2 + p + S * 3, ty1), fill=sc)
    d.rectangle((cx - tw // 2, ty1, cx + tw // 2, Hb * .8), fill=jitter(rng.choice(PALETTE), rng))  # shorts
    d.rectangle((cx - tw // 2.5, Hb * .8, cx + tw // 2.5, Hb), fill=skin)                           # legs
    layer = Image.new('L', (Wb, Hb), 0)                                                           # the number
    ny = ty0 + (ty1 - ty0) * rng.uniform(.38, .55)
    font = ImageFont.truetype(rng.choice(FONTS), int((ty1 - ty0) * rng.uniform(.45, .7)))
    ImageDraw.Draw(layer).text((cx + rng.randint(-S * 2, S * 2), ny), str(number), fill=255, font=font, anchor='mm')
    layer = layer.rotate(rng.uniform(-10, 10), center=(cx, ny))
    if rng.random() < .3:                                                                         # outline
        im.paste(jitter(rng.choice(PALETTE), rng), mask=Image.fromarray(cv2.dilate(np.array(layer), np.ones((S + 1, S + 1), np.uint8))))
    im.paste(numc, mask=layer)
    a = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2BGR)
    if rng.random() < .6:                                                                         # motion blur
        k = rng.randint(3, 4 * S); ker = np.zeros((k, k)); ker[k // 2, :] = 1 / k
        a = cv2.filter2D(a, -1, ker)
    a = cv2.resize(a, (W, H), interpolation=cv2.INTER_AREA)                                       # real size
    a = cv2.GaussianBlur(a, (3, 3), rng.uniform(.1, 1.0))
    a = np.clip(a + np.random.normal(0, rng.uniform(2, 8), a.shape), 0, 255).astype(np.uint8)    # noise
    _, enc = cv2.imencode('.jpg', a, [cv2.IMWRITE_JPEG_QUALITY, rng.randint(35, 85)])               # JPEG
    return cv2.imdecode(enc, 1), cv2.resize(np.array(layer), (W, H), interpolation=cv2.INTER_AREA) > 60

def sample_number(rng):
    return rng.randint(1, 9) if rng.random() < .3 else rng.randint(10, 99)

### 5.2 Digit patches
Each digit box is cut from the contrast map, stretched to 0–255, resized to fit 22 px (shape kept) and centred
in 28x28. Every digit then has the same size and brightness, always bright on dark.

In [ ]:
def normalize_patch(p, size=28, inner=22):
    p = p.astype(np.float32)
    p = (p - p.min()) / (p.max() - p.min() + 1e-6) * 255                      # contrast stretch
    h, w = p.shape; s = inner / max(h, w)
    r = cv2.resize(p, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    out = np.zeros((size, size), np.float32)
    y0, x0 = (size - r.shape[0]) // 2, (size - r.shape[1]) // 2
    out[y0:y0 + r.shape[0], x0:x0 + r.shape[1]] = r
    return out.astype(np.uint8)

def candidate_patches(sa, cand):
    return [normalize_patch(sa['dE'][b['y']:b['y'] + b['h'], b['x']:b['x'] + b['w']]) for b in cand['boxes']]

def process_torso(t):
    """Steps 2-4 on one torso (with optional deblurring from A4)."""
    if CFG['deblur']:
        t = deblur(t, CFG['deblur'])
    return candidates_for(t)

def build_synthetic_digits(n_players, seed=0):
    rng = random.Random(seed); np.random.seed(seed)
    X, y, used, t0 = [], [], 0, time.time()
    for i in range(n_players):
        n = sample_number(rng); s = str(n)
        img, nm = synth_player(n, rng)
        nm_t = upscale(torso_crop(nm.astype(np.uint8) * 255)) > 127               # where the number really is
        sa, cands = process_torso(upscale(torso_crop(img)))
        for c in cands:
            if len(c['boxes']) == len(s) and all(
                    nm_t[b['y']:b['y'] + b['h'], b['x']:b['x'] + b['w']][b['mask']].mean() > .5 for b in c['boxes']):
                X += candidate_patches(sa, c); y += [int(ch) for ch in s]; used += 1
                break
        if (i + 1) % 1000 == 0:
            print(f'{i+1}/{n_players} players, {used} usable, {len(y)} digits, {time.time()-t0:.0f}s')
    return np.array(X), np.array(y)

SYN_FILE = os.path.join(WORK, 'synthetic_digits.npz')
if not os.path.exists(SYN_FILE):
    X_syn, y_syn = build_synthetic_digits(N_SYNTH)
    np.savez(SYN_FILE, X=X_syn, y=y_syn)
d = np.load(SYN_FILE); X_syn, y_syn = d['X'], d['y']
print('synthetic digits:', len(y_syn), np.bincount(y_syn, minlength=10))
rng = random.Random(3)
show_grid([upscale(synth_player(sample_number(rng), rng)[0]) for _ in range(8)] + list(X_syn[:8]),
          [''] * 8 + list(y_syn[:8]), cols=8, size=1.4)

### 5.3 Three digit classifiers (all trained from scratch)
All share one interface, `predict_proba(list of 28x28 patches) -> (n, 10) probabilities`, so any of them can
be plugged into the reader. From simplest to most complex:

1. **Shape features + kNN / SVM** (the professor's baseline): from the binarized digit (Otsu), we measure
   * **holes** with contour hierarchy (0 → 1 hole, 8 → 2, 1 → none) and **where the hole is** (6 vs 9),
   * **aspect ratio** and **fill ratio**,
   * **Hu moments**: 7 numbers describing the shape, unchanged by size and position,
   * **3x3 zoning**: share of white pixels in each cell of a 3x3 grid.
2. **HOG + SVM**: Histogram of Oriented Gradients = which way the edges point in each 7x7 cell.
3. **Small CNN** designed and trained here from scratch (no pretrained weights).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from skimage.feature import hog

def shape_features(patch):
    """22 contour/shape features of the digit in a 28x28 patch."""
    _, b = cv2.threshold(patch, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (b > 0).astype(np.uint8)
    n, lbl, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n < 2:
        return np.zeros(22)
    m = (lbl == 1 + st[1:, cv2.CC_STAT_AREA].argmax()).astype(np.uint8)        # keep the main blob
    ys, xs = np.nonzero(m)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    crop = m[y0:y1, x0:x1]
    h, w = crop.shape
    cnts, hier = cv2.findContours(m, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    holes = [c for c, hh in zip(cnts, hier[0]) if hh[3] >= 0 and cv2.contourArea(c) >= 2]   # inner contours
    n_holes = min(len(holes), 2)
    hole_y = np.mean([cv2.boundingRect(c)[1] + cv2.boundingRect(c)[3] / 2 - y0 for c in holes]) / h if holes else -1
    hu = cv2.HuMoments(cv2.moments(crop, binaryImage=True)).ravel()
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)                          # log scale: values span decades
    zones = cv2.resize(crop.astype(np.float32), (3, 3), interpolation=cv2.INTER_AREA).ravel()
    return np.concatenate([np.eye(3)[n_holes], [hole_y, w / h, crop.mean()], hu, zones])

def hog_features(patch):
    return hog(patch, orientations=9, pixels_per_cell=(7, 7), cells_per_block=(2, 2))

class FeatureRecognizer:
    """Features (shape or HOG) -> scaler -> classifier (kNN or SVM)."""
    def __init__(self, features, clf):
        self.features = features
        self.model = make_pipeline(StandardScaler(), clf)
    def fit(self, X, y):
        self.model.fit(np.array([self.features(x) for x in X]), y); return self
    def predict_proba(self, patches):
        return self.model.predict_proba(np.array([self.features(p) for p in patches]))

### 5.4 Training digits, part 2: real digits harvested from dev players
We know every dev player's number. For each dev frame we pick the candidate that best matches it and keep its
digits if the match is confident. Two rounds (round 2 uses a classifier that already saw round-1 digits).
Hand-labelled players are excluded so the test images stay clean.

In [ ]:
def read_candidates(sa, cands, rec):
    """Digit probabilities for every candidate."""
    return [rec.predict_proba(candidate_patches(sa, c)) for c in cands]

def harvest_real_digits(pool, rec, min_conf=0.3):
    X, y, used, t0 = [], [], 0, time.time()
    for i, it in enumerate(pool):
        s = str(it['gt'])
        sa, cands = process_torso(get_torso(it['path']))
        best, best_p = None, 0.0
        for c in cands:
            if len(c['boxes']) != len(s): continue
            patches = candidate_patches(sa, c)
            P = rec.predict_proba(patches)
            p = float(np.prod([P[k, int(ch)] for k, ch in enumerate(s)]) ** (1 / len(s)))   # match to the true number
            if p > best_p: best, best_p = patches, p
        if best is not None and best_p >= min_conf:
            X += best; y += [int(ch) for ch in s]; used += 1
        if (i + 1) % 1000 == 0:
            print(f'{i+1}/{len(pool)} frames, {used} used, {len(y)} digits, {time.time()-t0:.0f}s')
    return np.array(X).reshape(-1, 28, 28), np.array(y, int)

REAL_FILE = os.path.join(WORK, 'real_digits.npz')
if not os.path.exists(REAL_FILE):
    held = {it['tid'] for it in labelled}
    pool = [it for it in items if it['split'] == 'dev' and it['gt'] >= 0 and it['usable'] and it['tid'] not in held]
    r0 = FeatureRecognizer(hog_features, SVC(C=5, gamma='scale', probability=True, random_state=SEED)).fit(X_syn, y_syn)
    X1, y1 = harvest_real_digits(pool, r0)
    r1 = FeatureRecognizer(hog_features, SVC(C=5, gamma='scale', probability=True, random_state=SEED)).fit(
        np.concatenate([X_syn, X1]), np.concatenate([y_syn, y1]))
    X_real, y_real = harvest_real_digits(pool, r1)
    np.savez(REAL_FILE, X=X_real, y=y_real)
d = np.load(REAL_FILE); X_real, y_real = d['X'], d['y']
print('real digits:', len(y_real), np.bincount(y_real, minlength=10))

### 5.5 Train and compare the classifiers on held-out digits
The same 85/15 split for every classifier. Held-out accuracy is shown separately for synthetic and real digits.

In [ ]:
def split_idx(n, frac=0.15, seed=SEED):
    p = np.random.RandomState(seed).permutation(n)
    return p[int(frac * n):], p[:int(frac * n)]

tr_s, te_s = split_idx(len(y_syn)); tr_r, te_r = split_idx(len(y_real))
X_tr = np.concatenate([X_syn[tr_s], X_real[tr_r]]); y_tr = np.concatenate([y_syn[tr_s], y_real[tr_r]])
X_te_syn, y_te_syn, X_te_real, y_te_real = X_syn[te_s], y_syn[te_s], X_real[te_r], y_real[te_r]

recognizers = {
    'shape+kNN': cached('rec_shape_knn', lambda: FeatureRecognizer(shape_features,
                        KNeighborsClassifier(n_neighbors=7, weights='distance')).fit(X_tr, y_tr)),
    'shape+SVM': cached('rec_shape_svm', lambda: FeatureRecognizer(shape_features,
                        SVC(C=5, gamma='scale', probability=True, random_state=SEED)).fit(X_tr, y_tr)),
    'HOG+SVM':   cached('rec_hog_svm', lambda: FeatureRecognizer(hog_features,
                        SVC(C=5, gamma='scale', probability=True, random_state=SEED)).fit(X_tr, y_tr)),
}

Optional: small CNN trained from scratch (runs on CPU in a few minutes; skipped if PyTorch is missing).

In [ ]:
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    class SmallCNN(nn.Module):
        """3 conv layers + 2 fully connected; designed and trained from scratch."""
        def __init__(self):
            super().__init__()
            self.c1, self.c2, self.c3 = nn.Conv2d(1, 32, 3, padding=1), nn.Conv2d(32, 64, 3, padding=1), nn.Conv2d(64, 64, 3, padding=1)
            self.fc1, self.fc2, self.drop = nn.Linear(64 * 7 * 7, 128), nn.Linear(128, 10), nn.Dropout(0.3)
        def forward(self, x):
            x = F.max_pool2d(F.relu(self.c1(x)), 2)          # 28 -> 14
            x = F.max_pool2d(F.relu(self.c2(x)), 2)          # 14 -> 7
            x = F.relu(self.c3(x))
            return self.fc2(self.drop(F.relu(self.fc1(x.flatten(1)))))

    class CNNRecognizer:
        def fit(self, X, y, epochs=15, bs=128):
            torch.manual_seed(SEED)
            Xt = torch.tensor(X, dtype=torch.float32).unsqueeze(1) / 255.
            yt = torch.tensor(y, dtype=torch.long)
            self.model = SmallCNN().to(DEVICE)
            opt = torch.optim.Adam(self.model.parameters(), 1e-3)
            for _ in range(epochs):
                self.model.train()
                for b in torch.randperm(len(yt)).split(bs):
                    dx, dy = np.random.randint(-2, 3, 2)                       # random shift augmentation
                    xb = torch.roll(Xt[b], shifts=(int(dy), int(dx)), dims=(2, 3)).to(DEVICE)
                    loss = F.cross_entropy(self.model(xb), yt[b].to(DEVICE))
                    opt.zero_grad(); loss.backward(); opt.step()
            self.model.cpu(); return self
        def predict_proba(self, patches):
            self.model.eval()
            with torch.no_grad():
                x = torch.tensor(np.array(patches), dtype=torch.float32).unsqueeze(1) / 255.
                return F.softmax(self.model(x), 1).numpy()

    recognizers['CNN'] = cached('rec_cnn', lambda: CNNRecognizer().fit(X_tr, y_tr))
except ImportError:
    print('PyTorch not installed: CNN skipped')

def digit_acc(rec, X, y):
    return (rec.predict_proba(list(X)).argmax(1) == y).mean()

print(f"{'classifier':12s} {'synthetic':>10s} {'real':>8s}   (held-out digit accuracy)")
for name, rec in recognizers.items():
    print(f"{name:12s} {digit_acc(rec, X_te_syn, y_te_syn):10.1%} {digit_acc(rec, X_te_real, y_te_real):8.1%}")

### 5.6 The reader: pick the candidate the classifier is most sure about
* Candidate confidence = **geometric mean** of its digit probabilities (fair to 1- and 2-digit numbers).
* A leading 0 or a lone 0 is not a valid number.
* A lone **"1"** is penalised (x0.7): arm edges and stripes look like 1s.

In [ ]:
def read_number(t, rec):
    """Returns (number or None, confidence, chosen candidate, predictions of all candidates)."""
    sa, cands = process_torso(t)
    best, preds = (None, 0.0, None), []
    for c, P in zip(cands, read_candidates(sa, cands, rec) if cands else []):
        digits = P.argmax(1)
        preds.append(int(''.join(map(str, digits))))
        if digits[0] == 0:
            continue
        conf = float(np.prod(P.max(1)))
        if CFG['geo_mean']:
            conf = conf ** (1 / len(digits))
        if len(digits) == 1 and digits[0] == 1:
            conf *= CFG['one_penalty']
        if conf > best[1]:
            best = (preds[-1], conf, c)
    return best[0], best[1], best[2], preds

def read_image(img, rec):
    """Read a full player crop: try every torso window in CFG['crops'], keep the most confident reading.
    Returns (number or None, confidence, chosen candidate, all predictions, torso used)."""
    best = None
    for top, bottom in CFG['crops']:
        t = upscale(torso_crop(img, top=top, bottom=bottom))
        num, conf, cand, preds = read_number(t, rec)
        if best is None or conf > best[1]:
            best = (num, conf, cand, preds + (best[3] if best else []), t)
        else:
            best = best[:3] + (best[3] + preds, best[4])
    return best

def readable_score(rec, items_=None):
    """Exact numbers read on the readable images + how often the right number was among the candidates."""
    items_ = readable if items_ is None else items_
    exact = ceiling = 0
    for it in items_:
        num, _, _, preds, _ = read_image(cv2.imread(it['path']), rec)
        exact += num == it['gt']; ceiling += it['gt'] in preds
    return exact, ceiling, len(items_)

print(f"{'classifier':12s} {'exact':>12s} {'ceiling':>12s}   (on the {len(readable)} readable images)")
for name, rec in recognizers.items():
    e, c, n = readable_score(rec)
    print(f"{name:12s} {e:5d} ({e/max(1,n):4.0%}) {c:5d} ({c/max(1,n):4.0%})")

## Step 6: Decide and evaluate
Below a confidence threshold **τ** the system answers **"unreadable"** instead of guessing (our backup plan).

* **Frame level** (hand-labelled images): coverage (answered), precision (correct when answering), abstain rate
  on unreadable images.
* **Player level** (all val/test players, no hand labels needed): each player's 6 frames vote, weighted by
  confidence; no confident frame = "unreadable" (-1). This is how SoccerNet scores the task.
* τ is chosen on **val** players; **test** is scored once at that τ.

In [ ]:
def read_all(items_, rec, label=''):
    """Reader on every frame: prediction + confidence (τ is applied later)."""
    rows, t0 = [], time.time()
    for i, it in enumerate(items_):
        img = cv2.imread(it['path'])
        num, conf = None, 0.0
        if img is not None and img.shape[0] >= MIN_H:              # height only (crops are narrow)
            num, conf, _, _, _ = read_image(img, rec)
        rows.append(dict(path=it['path'], tid=it['tid'], gt=it['gt'], pred=num, conf=conf,
                         h=it['h'], sharp=it['sharp'], legible=legible.get(it['path'])))
        if (i + 1) % 2000 == 0:
            print(f'  {label} {i+1}/{len(items_)} frames, {time.time()-t0:.0f}s')
    return rows

def frame_metrics(rows, tau):
    rd = [r for r in rows if r['legible'] is True]
    un = [r for r in rows if r['legible'] is False]
    ans = [r for r in rd if r['pred'] is not None and r['conf'] >= tau]
    ok = sum(r['pred'] == r['gt'] for r in ans)
    return dict(tau=tau, coverage=len(ans) / max(1, len(rd)), precision=ok / max(1, len(ans)),
                accuracy=ok / max(1, len(rd)),
                abstain_unreadable=sum(r['pred'] is None or r['conf'] < tau for r in un) / max(1, len(un)))

def player_vote(rows, tau):
    """One answer per player: confidence-weighted vote of its frames; nothing confident -> -1."""
    by = defaultdict(list)
    for r in rows:
        by[r['tid']].append(r)
    out = {}
    for tid, rs in by.items():
        votes = Counter()
        for r in rs:
            if r['pred'] is not None and r['conf'] >= tau:
                votes[r['pred']] += r['conf']
        out[tid] = (votes.most_common(1)[0][0] if votes else -1, rs[0]['gt'])
    return out

def player_metrics(rows, tau):
    v = player_vote(rows, tau)
    vis = [(p, g) for p, g in v.values() if g >= 0]
    inv = [(p, g) for p, g in v.values() if g < 0]
    answered = [(p, g) for p, g in vis if p != -1]
    return dict(tau=tau, players=len(v),
                overall=sum(p == g for p, g in v.values()) / max(1, len(v)),        # -1 counts as a class
                visible_acc=sum(p == g for p, g in vis) / max(1, len(vis)),
                visible_coverage=len(answered) / max(1, len(vis)),
                visible_precision=sum(p == g for p, g in answered) / max(1, len(answered)),
                abstain_on_no_number=sum(p == -1 for p, g in inv) / max(1, len(inv)))

def fmt(d):
    return ' | '.join(f'{k}={v:.2f}' if isinstance(v, float) else f'{k}={v}' for k, v in d.items())

TAUS = np.round(np.arange(0, 0.96, 0.05), 2)
val_items = [it for it in items if it['split'] == 'val']

### 6.1 All classifiers on val (choose τ) and test (final score)
Predictions are saved to Drive, so this is slow (~4 min per classifier on test) only the first time.

In [ ]:
results, rows_by = {}, {}
for name, rec in recognizers.items():
    key = name.replace('+', '_')
    rows_by[name] = dict(lab=cached(f'rows_{key}_lab{len(labelled)}', lambda: read_all(labelled, rec, 'labelled')),
                         val=cached(f'rows_{key}_val', lambda: read_all(val_items, rec, 'val')),
                         test=cached(f'rows_{key}_test', lambda: read_all(test_items, rec, 'test')))
    tau = max(TAUS, key=lambda t: player_metrics(rows_by[name]['val'], t)['overall'])
    results[name] = dict(tau=float(tau), readable=frame_metrics(rows_by[name]['lab'], tau),
                         val=player_metrics(rows_by[name]['val'], tau), test=player_metrics(rows_by[name]['test'], tau))
    print(f'{name}: τ={tau}  TEST {fmt(results[name]["test"])}')

no_number = np.mean([test_labels[t] < 0 for t in {it['tid'] for it in test_items}])
MAIN = max(results, key=lambda n: results[n]['val']['overall'])     # best classifier on VAL
TAU = results[MAIN]['tau']
print(f'\nmain classifier (best on val): {MAIN}, τ = {TAU}')

cols = [('overall', 'Overall'), ('visible_acc', 'Visible: read'), ('visible_coverage', 'Visible: answered'),
        ('visible_precision', 'Correct when answering'), ('abstain_on_no_number', 'No number: "unreadable"')]
print(f"\n{'TEST players':26s}" + ''.join(f'{n:>12s}' for n in results))
for k, label in cols:
    print(f'{label:26s}' + ''.join(f"{results[n]['test'][k]:12.2f}" for n in results))
baseline_label = '(always "unreadable")'
print(f'{baseline_label:26s}{no_number:12.2f}')
json.dump(dict(results=results, main=MAIN, always_unreadable=float(no_number)),
          open(os.path.join(WORK, 'step6_results.json'), 'w'), indent=2)

### 6.2 Fewer answers, more of them right (main classifier, hand-labelled frames)

In [ ]:
fm = [frame_metrics(rows_by[MAIN]['lab'], t) for t in TAUS]
for m in fm[::2]:
    print(fmt(m))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([m['coverage'] for m in fm], [m['precision'] for m in fm], 'o-')
for m in fm[::4]:
    ax[0].annotate(f"τ={m['tau']}", (m['coverage'], m['precision']), fontsize=8)
ax[0].set_xlabel('readable images answered'); ax[0].set_ylabel('answers correct'); ax[0].grid(alpha=.3)
ax[0].set_title('Readable images: answering less, but more accurately')
ax[1].plot(TAUS, [m['accuracy'] for m in fm], label='readable: correct')
ax[1].plot(TAUS, [m['abstain_unreadable'] for m in fm], label='unreadable: said "unreadable"')
ax[1].set_xlabel('confidence threshold τ'); ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_title('Choosing τ')
plt.tight_layout(); plt.savefig(os.path.join(FIGS, 'tau_tradeoff.png'), dpi=150); plt.show()

### 6.3 Accuracy vs image quality (readable images, τ = 0)

In [ ]:
rd = [r for r in rows_by[MAIN]['lab'] if r['legible']]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, key, name in ((ax[0], 'h', 'crop height (px)'), (ax[1], 'sharp', 'sharpness')):
    vals = np.array([r[key] for r in rd]); okk = np.array([r['pred'] == r['gt'] for r in rd])
    edges = np.percentile(vals, [0, 33, 67, 100])
    bins = np.digitize(vals, edges[1:-1])                         # 0, 1, 2 = lower / middle / upper third
    a.bar([f'{edges[i]:.0f}-{edges[i+1]:.0f}\n(n={(bins == i).sum()})' for i in range(3)],
          [okk[bins == i].mean() if (bins == i).any() else 0 for i in range(3)])
    a.set_ylim(0, 1); a.set_title(f'accuracy vs {name}')
plt.tight_layout(); plt.savefig(os.path.join(FIGS, 'accuracy_vs_quality.png'), dpi=150); plt.show()

## A1: Ablation study
Turn off **one** component at a time and measure the drop. Scored on the readable images (exact number) and on
val players (overall, at the main τ). Every row changes only one thing compared with the full pipeline.

In [ ]:
ABLATIONS = [
    ('Full pipeline', {}),
    ('Generic HSV grass mask', dict(grass='hsv')),
    ('Search only the shirt-colour region', dict(search='shirt')),
    ('No local-contrast masks', dict(local_mask=False)),
    ('No stricter thresholds', dict(strict=False)),
    ('No border clearing', dict(clear_border=False)),
    ('Only the best candidate (top-1)', dict(top_k=1)),
    ('Product instead of geometric mean', dict(geo_mean=False)),
    ('No lone-"1" penalty', dict(one_penalty=1.0)),
    ('Synthetic training digits only', 'SYN_ONLY'),
]
ABLATION_ON_VAL = True        # ~1 min per row on val (cached after the first run)

from sklearn.base import clone
rec_main = recognizers[MAIN]
abl = []
for name, change in ABLATIONS:
    rec, changes = rec_main, ({} if change == 'SYN_ONLY' else change)
    if change == 'SYN_ONLY':
        rec = cached(f'rec_{MAIN.replace("+", "_")}_synonly', lambda: (
            FeatureRecognizer(rec_main.features, clone(rec_main.model.steps[-1][1]))
            if isinstance(rec_main, FeatureRecognizer) else CNNRecognizer()).fit(X_syn[tr_s], y_syn[tr_s]))
    with use_cfg(**changes):
        e, c, n = readable_score(rec)
        key = re.sub(r'\W+', '_', name.lower())
        val = (player_metrics(cached(f'abl_{MAIN.replace("+", "_")}_{key}', lambda: read_all(val_items, rec, name)), TAU)['overall']
               if ABLATION_ON_VAL else float('nan'))
    abl.append((name, e / max(1, n), val))
    print(f'{name:38s} readable exact {e:3d}/{n} ({e/max(1,n):4.0%})   val overall {val:.2f}')

fig, ax = plt.subplots(figsize=(10, 4.2))
names_ = [a[0] for a in abl][::-1]
ax.barh(np.arange(len(abl)) - .2, [a[1] for a in abl][::-1], .4, label='readable images: exact')
ax.barh(np.arange(len(abl)) + .2, [a[2] for a in abl][::-1], .4, label='val players: overall')
ax.set_yticks(range(len(abl))); ax.set_yticklabels(names_, fontsize=8); ax.set_xlim(0, 1)
ax.axvline(abl[0][1], ls='--', lw=1, c='C0'); ax.axvline(abl[0][2], ls='--', lw=1, c='C1')
ax.set_title('Ablation: what each component contributes'); ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.savefig(os.path.join(FIGS, 'ablation.png'), dpi=150); plt.show()

## A2: Digit confusion matrix
Which digits get mistaken for which, on **held-out real digits**. Read each row as "true digit → predicted".
Confusions can be explained by shape (e.g. 3 vs 8 differ only by closing the left side).

Caveat: real digits were harvested with a classifier's help, so they are a bit easier than average.

In [ ]:
from sklearn.metrics import confusion_matrix

show = [n for n in ('shape+kNN', MAIN) if n in recognizers]
fig, axes = plt.subplots(1, len(show), figsize=(5.2 * len(show), 4.6))
for ax, name in zip(np.atleast_1d(axes), show):
    pred = recognizers[name].predict_proba(list(X_te_real)).argmax(1)
    cm = confusion_matrix(y_te_real, pred, labels=range(10))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(10):
        for j in range(10):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7, color='white' if cmn[i, j] > .5 else 'black')
    ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f'{name}: {(pred == y_te_real).mean():.0%} of {len(pred)} real digits')
    pairs = sorted(((cm[i, j], i, j) for i in range(10) for j in range(10) if i != j), reverse=True)[:5]
    print(f'{name} most common confusions:', ', '.join(f'{i}→{j} ({n}x)' for n, i, j in pairs if n))
plt.tight_layout(); plt.savefig(os.path.join(FIGS, 'confusion.png'), dpi=150); plt.show()

## A3: Pipeline walkthrough figure
One player image through every stage. Good for the presentation and for explaining any part of the pipeline.

In [ ]:
def walkthrough(it, rec=None, save=None):
    rec = rec or recognizers[MAIN]
    img = cv2.imread(it['path']); t = upscale(torso_crop(img))
    sa = shirt_analysis(t); masks = number_masks(sa); cands = segment_digits(sa, masks)
    num, conf, cand, _ = read_number(t, rec)
    to3 = lambda m: cv2.cvtColor((m * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    region = t.copy()
    cnts, _ = cv2.findContours(sa['region'].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(region, cnts, -1, (255, 255, 0), 2)
    mask = masks[cand['source']] if cand else np.zeros(t.shape[:2], bool)
    patches = candidate_patches(sa, cand) if cand else [np.zeros((28, 28), np.uint8)]
    panels = [(cv2.resize(img, (int(img.shape[1] * 128 / img.shape[0]), 128)), '0 Input crop'),
              (t, '1 Torso, upscaled'),
              (cv2.applyColorMap(np.clip(sa['dE'] * 2.5, 0, 255).astype(np.uint8), cv2.COLORMAP_INFERNO), '2 Contrast to shirt'),
              (region, '2 Search region'),
              (to3(mask), f"3 Number mask ({cand['source'] if cand else '-'})"),
              (draw_boxes(t, cand['boxes'] if cand else [], (0, 255, 0)), '4 Chosen digits'),
              (cv2.cvtColor(np.hstack(patches), cv2.COLOR_GRAY2BGR), '5 28x28 patches')]
    fig, ax = plt.subplots(1, len(panels) + 1, figsize=(2.1 * (len(panels) + 1), 3),
                           gridspec_kw=dict(width_ratios=[p.shape[1] / p.shape[0] for p, _ in panels] + [0.8]))
    for a, (p, title) in zip(ax, panels):
        a.imshow(p[..., ::-1]); a.set_title(title, fontsize=8); a.axis('off')
    ax[-1].axis('off')
    ax[-1].text(0.05, 0.5, f"6 Answer: {num if num is not None and conf >= TAU else 'unreadable'}\n"
                f"confidence {conf:.2f} (τ={TAU})\ntrue: {it['gt']}", fontsize=10, va='center')
    plt.tight_layout()
    if save:
        plt.savefig(os.path.join(FIGS, save), dpi=150)
    plt.show()

right = [it for it in readable if read_number(get_torso(it['path']), recognizers[MAIN])[0] == it['gt']]
wrong = [it for it in readable if it not in right]
for i, it in enumerate(right[:2]):
    walkthrough(it, save=f'walkthrough_correct_{i}.png')
if wrong:
    walkthrough(wrong[0], save='walkthrough_failure.png')

## A4: Deblurring before reading
Blur was the clearest accuracy factor. Two classic ways to sharpen before Steps 2–5:
* **Unsharp masking**: add back the difference between the image and a blurred copy (boosts edges).
* **Wiener deconvolution**: undo a known blur in the frequency domain. We assume horizontal **motion blur**
  (players run sideways) of length L px; K stops noise from exploding where the blur removed information.

The classifiers were trained on non-deblurred digits, so this tests deblurring as a plug-in step.

In [ ]:
def unsharp(img, sigma=2.0, amount=1.0):
    blur = cv2.GaussianBlur(img, (0, 0), sigma)
    return cv2.addWeighted(img, 1 + amount, blur, -amount, 0)

def wiener(img, L=7, K=0.01):
    """Wiener deconvolution with a horizontal motion-blur kernel of length L (per colour channel)."""
    psf = np.zeros(img.shape[:2], np.float32)
    psf[0, :L] = 1 / L
    psf = np.roll(psf, -(L // 2), axis=1)                        # centre the kernel at the origin
    Hf = np.fft.fft2(psf)
    G = np.conj(Hf) / (np.abs(Hf) ** 2 + K)                      # Wiener filter
    out = np.stack([np.real(np.fft.ifft2(np.fft.fft2(img[..., c].astype(np.float32)) * G)) for c in range(3)], 2)
    return np.clip(out, 0, 255).astype(np.uint8)

def deblur(img, how):
    return unsharp(img) if how == 'unsharp' else wiener(img) if how == 'wiener' else img

blurry = sorted(readable, key=lambda it: it['sharp'])[:6]
tiles = []
for it in blurry:
    t = get_torso(it['path'])
    tiles += [t, unsharp(t), wiener(t)]
show_grid(tiles, ['original', 'unsharp', 'Wiener'] * len(blurry), cols=6, size=1.6)

In [ ]:
sh = np.array([it['sharp'] for it in readable])
blur_third = [it for it in readable if it['sharp'] <= np.percentile(sh, 33)]
print(f"{'preprocessing':12s} {'all readable':>14s} {'blurriest third':>16s} {'val overall':>12s}")
for how in (None, 'unsharp', 'wiener'):
    with use_cfg(deblur=how):
        e, _, n = readable_score(recognizers[MAIN])
        eb, _, nb = readable_score(recognizers[MAIN], blur_third)
        val = player_metrics(cached(f'deblur_{MAIN.replace("+", "_")}_{how}', lambda: read_all(val_items, recognizers[MAIN], str(how))), TAU)['overall']
    print(f"{str(how):12s} {e:5d}/{n} ({e/max(1,n):4.0%}) {eb:7d}/{nb} ({eb/max(1,nb):4.0%}) {val:12.2f}")

# Part B: Improvements
Three add-ons, tested with the main classifier. Each is a switch, so the results above are unchanged.

| | Problem | Idea |
|---|---|---|
| B1 | Pale numbers on white kits are missed | **Chroma contrast**: compare only the colour channels (LAB a, b), ignore brightness |
| B2 | Accuracy drops for the largest players | **Multi-crop**: try 3 torso windows, keep the most confident reading |
| B3 | Frames without a visible number still vote | **Visibility classifier**: only frames judged "number visible" vote |

## B1 + B2: Reading improvements

**B1 Chroma contrast.** Shading on a white shirt changes mostly the **brightness** (L), while an orange number
differs in **colour** (a, b). A contrast map on a and b only makes pale coloured numbers stand out.

**B2 Multi-crop.** The fixed torso window (drop top 1/6, bottom 1/3) may cut the number on close-up players.
We also try a window reaching lower and one starting higher, and keep the reading with the highest confidence.

In [ ]:
CROPS3 = ((1/6, 1/3), (0.10, 0.20), (0.05, 0.40))     # default | taller (reaches lower) | higher

def white_kit(it):
    """True if the shirt colour found in Step 2 is near-white (bright, almost no colour)."""
    sa = shirt_analysis(get_torso(it['path']))
    return sa is not None and sa['shirt'][0] > 170 and np.hypot(sa['shirt'][1] - 128, sa['shirt'][2] - 128) < 12

white = [it for it in readable if white_kit(it)]
hs = np.array([it['h'] for it in readable])
large = [it for it in readable if it['h'] >= np.percentile(hs, 67)]
print(f'readable: {len(readable)} | white kits: {len(white)} | largest third: {len(large)}')

def pct(e, n):
    return f'{e:2d}/{n:<2d} ({e/max(1,n):4.0%})'

VARIANTS = [('Current pipeline', {}),
            ('B1 chroma contrast', dict(chroma_mask=True)),
            ('B2 multi-crop', dict(crops=CROPS3)),
            ('B1 + B2', dict(chroma_mask=True, crops=CROPS3))]
rec_main = recognizers[MAIN]
variant_scores = {}
print(f"\n{'variant':20s} {'all readable':>15s} {'white kits':>15s} {'largest third':>15s} {'val overall':>12s}")
for name, change in VARIANTS:
    with use_cfg(**change):
        e, _, n = readable_score(rec_main)
        ew, _, nw = readable_score(rec_main, white)
        el, _, nl = readable_score(rec_main, large)
        key = re.sub(r'\W+', '_', name.lower())
        val = player_metrics(cached(f'imp_{MAIN.replace("+", "_")}_{key}_val',
                                    lambda: read_all(val_items, rec_main, name)), TAU)['overall']
    variant_scores[name] = val
    print(f'{name:20s} {pct(e, n):>15s} {pct(ew, nw):>15s} {pct(el, nl):>15s} {val:12.2f}')

BEST_VARIANT = max(variant_scores, key=variant_scores.get)          # chosen on VAL, not test
BEST_CFG = dict(VARIANTS)[BEST_VARIANT]
print(f'\nbest on val: {BEST_VARIANT}')

In [ ]:
# Before / after on white kits: does the chroma map show the number?
if white:
    to3 = lambda m: cv2.cvtColor((m * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    panels = []
    for it in white[:8]:
        t = get_torso(it['path']); sa = shirt_analysis(t)
        chroma = np.linalg.norm(sa['lab'][..., 1:] - sa['shirt'][1:], axis=2)
        panels.append(np.hstack([t, cv2.applyColorMap(np.clip(sa['dE'] * 2.5, 0, 255).astype(np.uint8), cv2.COLORMAP_INFERNO),
                                 cv2.applyColorMap(np.clip(chroma * 5, 0, 255).astype(np.uint8), cv2.COLORMAP_INFERNO)]))
    show_panels(panels, [f"#{it['gt']}: torso | full contrast | colour-only contrast" for it in white[:8]])

Test score of the best variant (one run, cached):

In [ ]:
with use_cfg(**BEST_CFG):
    key = re.sub(r'\W+', '_', BEST_VARIANT.lower())
    rows_best_val = cached(f'imp_{MAIN.replace("+", "_")}_{key}_val', lambda: read_all(val_items, rec_main))
    rows_best_test = cached(f'imp_{MAIN.replace("+", "_")}_{key}_test', lambda: read_all(test_items, rec_main, 'test'))
TAU_B = max(TAUS, key=lambda t: player_metrics(rows_best_val, t)['overall'])
print(f'{BEST_VARIANT} (τ={TAU_B}) TEST:', fmt(player_metrics(rows_best_test, TAU_B)))
print(f'before            (τ={TAU}) TEST:', fmt(results[MAIN]['test']))

## B3: Visibility (legibility) classifier
A small classifier answers one question per frame: **is a number visible here?** Only "yes" frames vote.
Koshkina & Elder (2024) use the same idea with a deep network; ours uses simple features the pipeline already
computes, and a **logistic regression** or **random forest** trained from scratch.

**Training labels (dev frames only):**
* **visible** = hand-labelled readable dev frames (both label files),
* **not visible** = hand-labelled unreadable dev frames + every frame of a **-1** player (free labels).

More "visible" examples make it better: set `RUN_LABELLING = True` and `LABEL_SPLIT = 'dev'` in the setup cell,
label ~300 more frames, then rerun from Step 1.3. Those labels are saved to a separate file
(`legible_labels_b3.json`) and used **only here**, so the readable test set above stays unchanged.

In [ ]:
FEATURES = ['confidence', 'n_candidates', 'best_score', 'n_digits', 'digit_height', 'digit_contrast',
            'log_sharpness', 'crop_height']

def frame_row(it, rec):
    """Read one frame and keep the features B3 learns from."""
    img = cv2.imread(it['path'])
    num, conf, cand, preds = None, 0.0, None, []
    if img is not None and img.shape[0] >= MIN_H:
        num, conf, cand, preds, _ = read_image(img, rec)
    boxes = cand['boxes'] if cand else []
    f = [conf, len(preds), cand['score'] if cand else -1, len(boxes),
         np.mean([b['h'] for b in boxes]) / TORSO_H if boxes else 0,
         np.mean([b['contrast'] for b in boxes]) if boxes else 0,
         np.log10(it['sharp'] + 1), it['h']]
    return dict(path=it['path'], tid=it['tid'], gt=it['gt'], pred=num, conf=conf, feats=f,
                h=it['h'], sharp=it['sharp'], legible=legible.get(it['path']))

def visible_label(r):
    """Training label for B3: hand label from either file, else 'not visible' for -1 players, else unknown."""
    lab = legible_b3.get(r['path'], legible.get(r['path']))
    return int(lab) if lab is not None else (0 if r['gt'] < 0 else None)

def frame_rows(items_, rec, label=''):
    t0, out = time.time(), []
    for i, it in enumerate(items_):
        out.append(frame_row(it, rec))
        if (i + 1) % 2000 == 0:
            print(f'  {label} {i+1}/{len(items_)} frames, {time.time()-t0:.0f}s')
    return out

dev_items = [it for it in items if it['split'] == 'dev']
key = re.sub(r'\W+', '_', BEST_VARIANT.lower())
with use_cfg(**BEST_CFG):
    rows_dev = cached(f'b3_{MAIN.replace("+", "_")}_{key}_dev', lambda: frame_rows(dev_items, rec_main, 'dev'))
    rows_val3 = cached(f'b3_{MAIN.replace("+", "_")}_{key}_val', lambda: frame_rows(val_items, rec_main, 'val'))
    rows_test3 = cached(f'b3_{MAIN.replace("+", "_")}_{key}_test', lambda: frame_rows(test_items, rec_main, 'test'))

# training set: dev frames with a known answer to "is a number visible?"
X_leg, y_leg = [], []
for r in rows_dev:
    lab = visible_label(r)
    if lab is not None:
        X_leg.append(r['feats']); y_leg.append(lab)
X_leg, y_leg = np.array(X_leg), np.array(y_leg)
print(f'visibility training set: {len(y_leg)} frames, {y_leg.sum()} visible, {(y_leg == 0).sum()} not visible')
if y_leg.sum() < 60:
    print('  few "visible" examples: label more dev frames for a better classifier (see above)')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

models = {
    'logistic regression': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000)),
    'random forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced', random_state=SEED),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
for name, m in models.items():
    auc = cross_val_score(m, X_leg, y_leg, cv=cv, scoring='roc_auc')
    print(f'{name:20s} 5-fold AUC {auc.mean():.2f} ± {auc.std():.2f}   (0.5 = guessing, 1.0 = perfect)')
    m.fit(X_leg, y_leg)

# which features matter (explainable!)
coef = models['logistic regression'][-1].coef_[0]
imp = models['random forest'].feature_importances_
print(f"\n{'feature':16s} {'LR weight':>10s} {'RF importance':>14s}")
for f, c, i in sorted(zip(FEATURES, coef, imp), key=lambda x: -x[2]):
    print(f'{f:16s} {c:10.2f} {i:14.2f}')

**Gated vote:** a frame votes only if P(visible) ≥ λ **and** confidence ≥ τ. λ and τ are chosen on val players;
test is scored once.

In [ ]:
def gated_metrics(rows, model, lam, tau):
    p = model.predict_proba(np.array([r['feats'] for r in rows]))[:, 1]
    gated = [dict(r, conf=r['conf'] if pv >= lam else 0.0) for r, pv in zip(rows, p)]   # gated frames don't vote
    return player_metrics(gated, tau)

best = None
for mname, m in models.items():
    for lam in (0.0, 0.3, 0.5, 0.7, 0.85):
        for tau in TAUS:
            v = gated_metrics(rows_val3, m, lam, tau)['overall']
            if best is None or v > best[0]:
                best = (v, mname, lam, tau)
_, LEG_MODEL, LAM, TAU_G = best
print(f'chosen on val: {LEG_MODEL}, λ = {LAM}, τ = {TAU_G}')

res_gate = gated_metrics(rows_test3, models[LEG_MODEL], LAM, TAU_G)
res_nogate = player_metrics(rows_test3, TAU_B)
print(f"\n{'TEST players':26s} {'no gate':>10s} {'with gate':>10s}")
for k, label in cols:
    print(f'{label:26s} {res_nogate[k]:10.2f} {res_gate[k]:10.2f}')

improvements = dict(variant=BEST_VARIANT, tau=float(TAU_B), test_no_gate=res_nogate,
                    gate=dict(model=LEG_MODEL, lam=LAM, tau=float(TAU_G), test=res_gate))
json.dump(improvements, open(os.path.join(WORK, 'partB_results.json'), 'w'), indent=2)

In [ ]:
wk = {it['path'] for it in white}
print(sum(it['path'] in wk for it in large), 'of', len(large), 'largest-third readable images are white kits')

In [ ]:
nonwhite_large = [it for it in large if it['path'] not in wk]
for it in nonwhite_large[:6]:
    walkthrough(it)

## B4: Separating touching digits with erosion
On close-up players the digits are big, thick and motion-blurred, so the two digits **touch** and Step 4 sees one blob
(e.g. "23", "33", "34" read as a single digit). **Erosion** shrinks every white blob by one pixel per step; the thin
bridge between two digits disappears first, so they become two separate blobs.

We **add** eroded copies of every Step 3 mask (1, 2 or 3 erosion steps with a 3x3 kernel) instead of replacing them,
so the reader can still pick the un-eroded version when that reads better. Scored like B1/B2; the best setting is
chosen on val.

In [ ]:
CFG.setdefault('erode', 0)          # B4: 0 = off, n = also add masks eroded n times
_number_masks_base = number_masks if '_number_masks_base' not in globals() else _number_masks_base

def number_masks(sa):
    """Step 3 masks, plus (B4) eroded copies that separate touching digits."""
    out = _number_masks_base(sa)
    if CFG['erode']:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        for name, m in list(out.items()):
            er = cv2.erode(m.astype(np.uint8), k, iterations=CFG['erode']) > 0   # thin bridges vanish first
            out[f'{name}_er{CFG["erode"]}'] = er
    return out

In [ ]:
wk = {it['path'] for it in white}
nonwhite_large = [it for it in large if it['path'] not in wk]

B4_VARIANTS = [('Current pipeline', {}), ('B4 erode x1', dict(erode=1)),
               ('B4 erode x2', dict(erode=2)), ('B4 erode x3', dict(erode=3))]
b4_val = {}
print(f"{'variant':18s} {'all readable':>15s} {'largest third':>15s} {'large, not white':>17s} {'val overall':>12s}")
for name, change in B4_VARIANTS:
    with use_cfg(**change):
        e, _, n = readable_score(rec_main)
        el, _, nl = readable_score(rec_main, large)
        en, _, nn = readable_score(rec_main, nonwhite_large)
        key = re.sub(r'\W+', '_', name.lower())
        rows_v = cached(f'b4_{MAIN.replace("+", "_")}_{key}_val', lambda: read_all(val_items, rec_main, name))
    b4_val[name] = player_metrics(rows_v, TAU)['overall']
    print(f'{name:18s} {pct(e, n):>15s} {pct(el, nl):>15s} {pct(en, nn):>17s} {b4_val[name]:12.2f}')

B4_BEST = max(b4_val, key=b4_val.get)                     # chosen on VAL
print(f'\nbest on val: {B4_BEST}')
if B4_BEST != 'Current pipeline':
    with use_cfg(**dict(B4_VARIANTS)[B4_BEST]):
        key = re.sub(r'\W+', '_', B4_BEST.lower())
        rv = cached(f'b4_{MAIN.replace("+", "_")}_{key}_val', lambda: read_all(val_items, rec_main))
        rt = cached(f'b4_{MAIN.replace("+", "_")}_{key}_test', lambda: read_all(test_items, rec_main, 'test'))
    tau_b4 = max(TAUS, key=lambda t: player_metrics(rv, t)['overall'])
    print(f'{B4_BEST} (τ={tau_b4}) TEST:', fmt(player_metrics(rt, tau_b4)))
    print(f'before          (τ={TAU}) TEST:', fmt(results[MAIN]['test']))

In [ ]:
# Chosen digit boxes without and with erosion (green = right number, red = wrong)
level = dict(B4_VARIANTS)[B4_BEST].get('erode', 2) or 2
panels, titles = [], []
for it in nonwhite_large[:8]:
    img = cv2.imread(it['path']); row = []
    for e in (0, level):
        with use_cfg(erode=e):
            num, conf, cand, _, t = read_image(img, rec_main)
        col = (0, 255, 0) if num == it['gt'] else (0, 0, 255)
        row.append(draw_boxes(t, cand['boxes'] if cand else [], col))
        titles_part = f"{num} ({conf:.2f})"
        row_title = titles_part if e == 0 else row_title + f"  →  {titles_part}"
    panels.append(np.hstack(row)); titles.append(f"#{it['gt']}: {row_title}")
fig = plt.figure(figsize=(4 * 3.2, math.ceil(len(panels) / 4) * 2.6))
for i, p in enumerate(panels):
    plt.subplot(math.ceil(len(panels) / 4), 4, i + 1); plt.imshow(p[..., ::-1]); plt.title(titles[i], fontsize=8); plt.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(FIGS, 'b4_erosion.png'), dpi=150); plt.show()

## Hand-off to Muzammil's part
`read_from_frame` takes a full match frame and one player box `(x0, y0, x1, y1)` in pixels from his detector.
Convert `(x, y, w, h)` boxes first: `x1 = x + w`, `y1 = y + h`. `read_player` votes over several crops of the
same player. Both use the best reading variant from Part B (chosen on val).

In [ ]:
def read_from_frame(frame_bgr, bbox_xyxy, rec=None, tau=None):
    """Jersey number of one detected player, or None = 'unreadable'. Returns (number, confidence)."""
    rec, tau = rec or recognizers[MAIN], TAU_B if tau is None else tau
    x0, y0, x1, y1 = map(int, bbox_xyxy)
    crop = frame_bgr[max(0, y0):y1, max(0, x0):x1]
    if crop.size == 0 or crop.shape[0] < MIN_H:
        return None, 0.0
    with use_cfg(**BEST_CFG):
        num, conf, _, _, _ = read_image(crop, rec)
    return (num if num is not None and conf >= tau else None), conf

def read_player(crops_bgr, rec=None, tau=None):
    """Several crops of the SAME player -> confidence-weighted vote (None if nothing confident)."""
    rec, tau = rec or recognizers[MAIN], TAU_B if tau is None else tau
    votes = Counter()
    for c in crops_bgr:
        if c is None or c.shape[0] < MIN_H:
            continue
        with use_cfg(**BEST_CFG):
            num, conf, _, _, _ = read_image(c, rec)
        if num is not None and conf >= tau:
            votes[num] += conf
    return votes.most_common(1)[0][0] if votes else None

# quick self-test: a test image placed in a fake 'frame'
it = test_items[0]; crop = cv2.imread(it['path'])
frame = np.zeros((crop.shape[0] + 40, crop.shape[1] + 40, 3), np.uint8); frame[20:-20, 20:-20] = crop
print('read_from_frame:', read_from_frame(frame, (20, 20, 20 + crop.shape[1], 20 + crop.shape[0])), '| true:', it['gt'])

# Part C: Next steps from the presentation
Two more improvements, built the same way as Part B (switches, chosen on val, test scored once):

| | Problem | Idea |
|---|---|---|
| B5 | Touching digits: B4's erosion separates them but leaves **thinner** digits than the ones the classifiers were trained on | **Erosion only to find one seed per digit, then watershed grows the seeds back** to full stroke thickness |
| B6 | Only ~6 frames per dev player were used to harvest real digits | Harvest from **4x more frames per labelled player**, keep the player id of every digit, **augment**, and measure on **held-out players** |

At the end, **Both halves together** runs Muzammil's player detector and this reader on the same full match frames.

## B5: Touching digits with erosion seeds + watershed
B4 adds eroded copies of the number mask. That separates touching digits, but every stroke also gets thinner, so the
28x28 patch looks different from the training digits. B5 keeps the good part of erosion and undoes the bad part:

1. Take a blob that is too wide for one digit (width/height > 0.7, the same blobs Step 4 tries to cut).
2. **Erode** it step by step (3x3 ellipse, at most 4 steps) until it falls apart into **two big pieces**. These are the
   **seeds**, one inside each digit (the thin bridge disappears first).
3. **Watershed** (marker-controlled): the seeds "flood" the original, un-eroded blob over the inverted **distance
   transform** (the distance of each pixel to the background, high in the middle of a stroke). Each pixel goes to the seed
   that reaches it first, and the two floods meet at the narrow bridge.
4. Each digit keeps its **full stroke thickness**. If no two seeds appear, Step 4's projection cut is used as before.

> Assessment notes: *distance transform* = distance of every foreground pixel to the nearest background pixel;
> *watershed* = region growing from markers in order of a relief, borders form where regions meet; *why seeds from
> erosion?* thin bridges disappear before the thick digit bodies, so each surviving piece sits inside one digit.

In [ ]:
CFG.setdefault('split', 'profile')     # B5: 'profile' = Step 4 projection cut | 'watershed' = erosion seeds + watershed

def split_watershed(c, max_erode=4):
    """Erode until the blob falls apart into 2 seeds, then grow the seeds back over the full blob with watershed.
    Returns 2 digit dicts (same format as split_wide), or None if the blob doesn't separate."""
    m = np.pad(c['mask'].astype(np.uint8), 2)                       # padding: erosion and watershed need a border
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    seeds = None
    for it in range(1, max_erode + 1):
        er = cv2.erode(m, k, iterations=it)
        if er.sum() == 0:
            return None
        n, lbl, st, _ = cv2.connectedComponentsWithStats(er, connectivity=8)
        big = [i for i in range(1, n) if st[i, cv2.CC_STAT_AREA] >= 0.15 * er.sum()]   # ignore crumbs
        if len(big) >= 2:
            seeds = sorted(big, key=lambda i: -st[i, cv2.CC_STAT_AREA])[:2]
            break
    if seeds is None:
        return None
    markers = np.zeros(m.shape, np.int32)
    markers[m == 0] = 1                                             # 1 = background
    for j, i in enumerate(seeds):
        markers[lbl == i] = j + 2                                   # 2, 3 = the two digit seeds
    dist = cv2.distanceTransform(m, cv2.DIST_L2, 3)
    relief = 255 - cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)   # stroke centres = valleys
    cv2.watershed(cv2.cvtColor(relief, cv2.COLOR_GRAY2BGR), markers)                     # -1 = border between digits
    parts = []
    for j in (2, 3):
        sub = (markers == j) & (m > 0)
        ys, xs = np.where(sub)
        if len(ys) == 0:
            return None
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        parts.append(dict(c, x=c['x'] + int(x0) - 2, y=c['y'] + int(y0) - 2, w=int(x1 - x0), h=int(y1 - y0),
                          cx=c['x'] + float(xs.mean()) - 2, mask=sub[y0:y1, x0:x1]))
    parts.sort(key=lambda p: p['x'])
    if min(p['h'] for p in parts) < 0.5 * c['h'] or min(p['w'] for p in parts) < 2:   # both must be digit-sized
        return None
    return parts

_split_wide_base = split_wide if '_split_wide_base' not in globals() else _split_wide_base

def split_wide(c):
    """Step 4 cut, or (B5) erosion seeds + watershed first, falling back to the projection cut."""
    if CFG['split'] == 'watershed':
        parts = split_watershed(c)
        if parts:
            return parts
    return _split_wide_base(c)

In [ ]:
B5_VARIANTS = [('Current pipeline', {}),
               ('B5 watershed split', dict(split='watershed'))]
if B4_BEST != 'Current pipeline':
    B5_VARIANTS += [(B4_BEST, dict(B4_VARIANTS)[B4_BEST]),
                    ('B5 + ' + B4_BEST, dict(dict(B4_VARIANTS)[B4_BEST], split='watershed'))]

b5_val = {}
print(f"{'variant':24s} {'all readable':>15s} {'largest third':>15s} {'large, not white':>17s} {'val overall':>12s}")
for name, change in B5_VARIANTS:
    with use_cfg(**change):
        e, _, n = readable_score(rec_main)
        el, _, nl = readable_score(rec_main, large)
        en, _, nn = readable_score(rec_main, nonwhite_large)
        key = re.sub(r'\W+', '_', name.lower())
        rows_v = cached(f'b5_{MAIN.replace("+", "_")}_{key}_val', lambda: read_all(val_items, rec_main, name))
    b5_val[name] = player_metrics(rows_v, TAU)['overall']
    print(f'{name:24s} {pct(e, n):>15s} {pct(el, nl):>15s} {pct(en, nn):>17s} {b5_val[name]:12.2f}')

B5_BEST = max(b5_val, key=b5_val.get)                         # chosen on VAL
B5_CFG = dict(B5_VARIANTS)[B5_BEST]
print(f'\nbest on val: {B5_BEST}')

## B6: More real training digits from labelled players
**Why.** Real digits are much more useful than synthetic ones (A1: "synthetic training digits only" is the biggest drop),
but Step 5.4 harvested them from only the 6 frames per player that Step 0.2 extracted, although every player clip has
hundreds of frames and **a label for the whole clip**.

**What changes.**
1. Extract **`HARVEST_FRAMES` frames per dev player** (default 24, 4x more) from the train zip, only for dev players
   (val and test players are never touched, nor the hand-labelled players).
2. Harvest exactly like Step 5.4 (the candidate that best matches the known number, if the match is confident), but
   **keep the player id** of every digit.
3. **Augment** the real digits: small rotation (±8°), scale (0.9-1.1), shift (±2 px) and blur, so the classifier sees
   the same digit in slightly different ways (a classic way to get more out of few examples).
4. **A fair test.** The 96% "real digit" accuracy of HOG+SVM in Step 5.5 is too optimistic for two reasons: the digits were
   harvested *with an HOG+SVM* (only digits it already agreed with were kept), and the held-out digits were split at
   random, so the same player (and nearly identical frames) sat on both sides. Here the held-out digits come from
   **players that no classifier was trained on**.

The comparison trains the **same classifier type** (`MAIN`) on: synthetic + harvest from the old 6 frames per player,
vs synthetic + harvest from `HARVEST_FRAMES` frames per player + augmentation. The real test is the full pipeline on
val (choose) and test (once).

In [ ]:
HARVEST_FRAMES = 24            # frames per dev player used for harvesting (Step 0.2 used 6)
AUG_COPIES = 2                 # augmented copies of every real training digit
EXTRA = os.path.join(LOCAL, 'extra_frames')

held = {it['tid'] for it in labelled}                                   # hand-labelled players stay out
dev_tids = sorted({it['tid'] for it in items if it['split'] == 'dev' and it['gt'] >= 0 and it['tid'] not in held})

def extract_frames_for(zip_path, out_dir, tids, k):
    """Extract k evenly spaced frames for each of the given players only."""
    tids = set(tids)
    with zipfile.ZipFile(zip_path) as z:
        by_t = defaultdict(list)
        for n in z.namelist():
            m = re.search(r'images/([^/]+)/[^/]+\.jpg$', n)
            if m and m.group(1) in tids:
                by_t[m.group(1)].append(n)
        keep = []
        for files in by_t.values():
            files.sort()
            keep += [files[i] for i in np.linspace(0, len(files) - 1, min(k, len(files))).round().astype(int)]
        for n in keep:
            z.extract(n, out_dir)
    print(f'extracted {len(keep)} frames for {len(by_t)} players')

train_zip = os.path.join(DRIVE_DATA, 'jersey-2023', 'train.zip')
if not os.path.exists(EXTRA) and os.path.exists(train_zip):
    extract_frames_for(train_zip, EXTRA, dev_tids, HARVEST_FRAMES)

def frames_of(tid):
    paths = sorted(glob.glob(os.path.join(EXTRA, '**', 'images', tid, '*.jpg'), recursive=True))
    return paths if paths else [it['path'] for it in items if it['tid'] == tid]   # fallback: the 6 frames

pool_old = [it for it in items if it['split'] == 'dev' and it['tid'] in set(dev_tids) and it['usable']]
pool_new = []
for t in dev_tids:
    for p in frames_of(t):
        img = cv2.imread(p)
        if img is not None and img.shape[0] >= MIN_H:
            pool_new.append(dict(path=p, gt=train_labels[t], tid=t))
print(f'harvest pool: {len(pool_old)} frames (6 per player) -> {len(pool_new)} frames ({HARVEST_FRAMES} per player), '
      f'{len(dev_tids)} players')

In [ ]:
def harvest_with_ids(pool, rec, min_conf=0.3):
    """Step 5.4 harvest, but every digit also keeps its player id."""
    X, y, tids, t0 = [], [], [], time.time()
    for i, it in enumerate(pool):
        s = str(it['gt'])
        sa, cands = process_torso(get_torso(it['path']))
        best, best_p = None, 0.0
        for c in cands:
            if len(c['boxes']) != len(s):
                continue
            patches = candidate_patches(sa, c)
            P = rec.predict_proba(patches)
            p = float(np.prod([P[k, int(ch)] for k, ch in enumerate(s)]) ** (1 / len(s)))
            if p > best_p:
                best, best_p = patches, p
        if best is not None and best_p >= min_conf:
            X += best; y += [int(ch) for ch in s]; tids += [it['tid']] * len(s)
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(pool)} frames, {len(y)} digits, {time.time()-t0:.0f}s')
    return np.array(X, np.uint8).reshape(-1, 28, 28), np.array(y, int), np.array(tids)

# The harvesting classifier is the same for both pools: HOG+SVM on synthetic + the Step 5.4 real digits (round 2)
harvester = cached('b6_harvester', lambda: FeatureRecognizer(hog_features, SVC(C=5, gamma='scale', probability=True,
                   random_state=SEED)).fit(np.concatenate([X_syn, X_real]), np.concatenate([y_syn, y_real])))
H_OLD = cached('b6_harvest_6', lambda: harvest_with_ids(pool_old, harvester))
H_NEW = cached(f'b6_harvest_{HARVEST_FRAMES}', lambda: harvest_with_ids(pool_new, harvester))
for name, (X_, y_, t_) in (('6 frames/player', H_OLD), (f'{HARVEST_FRAMES} frames/player', H_NEW)):
    print(f'{name:18s}: {len(y_):6d} real digits from {len(set(t_))} players  {np.bincount(y_, minlength=10)}')

In [ ]:
def augment(patch, rng):
    """Small random rotation, scale, shift and blur of a 28x28 digit patch."""
    M = cv2.getRotationMatrix2D((14, 14), rng.uniform(-8, 8), rng.uniform(0.9, 1.1))
    M[:, 2] += (rng.randint(-2, 2), rng.randint(-2, 2))
    out = cv2.warpAffine(patch, M, (28, 28), flags=cv2.INTER_LINEAR, borderValue=0)
    if rng.random() < 0.5:
        out = cv2.GaussianBlur(out, (3, 3), rng.uniform(0.3, 1.0))
    return out

# Held-out PLAYERS (15% of the harvested dev players): never used to train any B6 classifier
rs = np.random.RandomState(SEED)
all_t = np.array(sorted(set(H_NEW[2]) | set(H_OLD[2])))
test_t = set(rs.choice(all_t, max(1, int(0.15 * len(all_t))), replace=False))
X_ho, y_ho = H_NEW[0][np.isin(H_NEW[2], list(test_t))], H_NEW[1][np.isin(H_NEW[2], list(test_t))]
print(f'held-out: {len(y_ho)} digits from {len(test_t)} players')

def train_set(H, aug):
    keep = ~np.isin(H[2], list(test_t))
    Xr, yr = H[0][keep], H[1][keep]
    if aug:
        rng = random.Random(SEED)
        Xa = np.array([augment(x, rng) for x in Xr for _ in range(AUG_COPIES)]).reshape(-1, 28, 28)
        Xr, yr = np.concatenate([Xr, Xa]), np.concatenate([yr, np.repeat(yr, AUG_COPIES)])
    return np.concatenate([X_syn[tr_s], Xr]), np.concatenate([y_syn[tr_s], yr])

MAX_SVM = 40000            # SVM training time grows fast with the number of examples
def new_recognizer(X, y):
    if isinstance(rec_main, FeatureRecognizer):
        if len(y) > MAX_SVM:
            idx = np.random.RandomState(SEED).choice(len(y), MAX_SVM, replace=False)
            X, y = X[idx], y[idx]
        return FeatureRecognizer(rec_main.features, clone(rec_main.model.steps[-1][1])).fit(X, y)
    return CNNRecognizer().fit(X, y)

B6_RECS = {
    '6 frames (as before)': cached(f'b6_rec_{MAIN.replace("+", "_")}_old', lambda: new_recognizer(*train_set(H_OLD, False))),
    f'{HARVEST_FRAMES} frames + augmentation': cached(f'b6_rec_{MAIN.replace("+", "_")}_newaug', lambda: new_recognizer(*train_set(H_NEW, True))),
}
print(f"\n{'training digits':32s} {'held-out players':>17s} {'synthetic':>10s}")
for name, rec in B6_RECS.items():
    print(f'{name:32s} {digit_acc(rec, X_ho, y_ho):17.1%} {digit_acc(rec, X_te_syn, y_te_syn):10.1%}')

In [ ]:
# The full pipeline with each B6 classifier: readable images + val players (choose), then test once for the best
b6_val, b6_rows = {}, {}
print(f"{'classifier':32s} {'all readable':>15s} {'val overall':>12s}")
for name, rec in B6_RECS.items():
    with use_cfg(**BEST_CFG):
        e, _, n = readable_score(rec)
        key = re.sub(r'\W+', '_', name.lower())
        rows_v = cached(f'b6_{MAIN.replace("+", "_")}_{key}_val', lambda: read_all(val_items, rec, name))
    tau_ = max(TAUS, key=lambda t: player_metrics(rows_v, t)['overall'])
    b6_val[name] = (player_metrics(rows_v, tau_)['overall'], float(tau_))
    print(f'{name:32s} {pct(e, n):>15s} {b6_val[name][0]:12.2f}  (τ={tau_})')

B6_BEST = max(b6_val, key=lambda n: b6_val[n][0])
print(f'\nbest on val: {B6_BEST}')

# Part III: Both halves together
Muzammil's improved detector (Part I, 8A + 8B) finds the people in the **same 60 GSR frames** loaded in Part I, and
Sidra's improved reader (Part II, B5 + the best B6 classifier) reads the number in each box. GSR labels most players
with their jersey number, so for every labelled player with a number we count where the number gets lost:
**not detected** → **detected but "unreadable"** → **wrong number** → **correct**. Reading the **ground-truth box**
instead of our own box shows how much is lost because of the detector.

In [ ]:
# The reader with the Part C improvements: best B6 classifier, the B5 setting, and its τ chosen on val
REC, TAU_USED = B6_RECS[B6_BEST], b6_val[B6_BEST][1]

def read_box(frame, box):
    """Number read in one (x, y, w, h) box of a full frame, or None = 'unreadable'."""
    x, y, w, h = (int(v) for v in box)
    with use_cfg(**B5_CFG):
        return read_from_frame(frame, (x, y, x + w, y + h), rec=REC, tau=TAU_USED)[0]

counts = Counter()          # where each labelled number ends up, with our detector's boxes
counts_gt = Counter()       # the same reader on the ground-truth boxes
examples = []
for f in frames:                                              # Part I's frames, already in memory
    boxes = find_people_v2(f["image"], "robust", True)[3]     # Part I, 8A + 8B
    drawn = []
    for g in f["boxes"]:
        jersey = str(g.get("jersey") or "")
        if g["role"] not in ("player", "goalkeeper") or not jersey.isdigit():
            continue
        gt_num, gbox = int(jersey), (g["x"], g["y"], g["w"], g["h"])
        num_gt = read_box(f["image"], gbox)
        counts_gt["correct" if num_gt == gt_num else "unreadable" if num_gt is None else "wrong"] += 1
        best = max(boxes, key=lambda b: iou(gbox, b), default=None)
        if best is None or iou(gbox, best) < 0.3:
            counts["not detected"] += 1
            continue
        num = read_box(f["image"], best)
        counts["correct" if num == gt_num else "unreadable" if num is None else "wrong"] += 1
        drawn.append((best, num, gt_num))
    if len(examples) < 2 and drawn:
        examples.append((f["image"], drawn))

total = sum(counts.values())
print(f"labelled players with a number: {total}\n")
print(f"{'':16s} {'our boxes':>10s} {'GT boxes':>10s}")
for k in ("not detected", "unreadable", "wrong", "correct"):
    print(f"{k:16s} {counts[k] / max(1, total):10.2f} {counts_gt[k] / max(1, total):10.2f}")
answered = counts["correct"] + counts["wrong"]
print(f"\ncorrect when answering (our boxes): {counts['correct'] / max(1, answered):.2f}")

In [ ]:
# Two frames: our boxes with the number read (green = right, red = wrong, grey = "unreadable")
for image, drawn in examples:
    v = image.copy()
    for (x, y, w, h), num, gt_num in drawn:
        col = (128, 128, 128) if num is None else (0, 200, 0) if num == gt_num else (0, 0, 255)
        cv2.rectangle(v, (x, y), (x + w, y + h), col, 3)
        cv2.putText(v, "?" if num is None else str(num), (x, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 1.0, col, 2)
    plt.figure(figsize=(14, 8)); plt.imshow(v[..., ::-1]); plt.axis("off"); plt.show()